# PROJECT SIGNAL
## The Hidden Economics of Customer Decisions

### Project Overview

Project SIGNAL is a customer analytics research project investigating purchasing behaviour, repeat purchases, discount economics, customer profitability and retention opportunities.

Using the Sample Superstore dataset (2015–2018), the project transforms transaction-level data into customer-level insights through exploratory data analysis, statistical modelling, RFM segmentation and predictive analytics.

### Research Objectives

1. Identify purchasing patterns associated with repeat customers.
2. Investigate the relationship between discounts and profitability.
3. Segment customers according to recency, frequency and monetary value.
4. Identify potentially valuable customers requiring re-engagement.
5. Evaluate whether historical purchasing information can predict future repeat purchases.
6. Develop evidence-based customer retention recommendations.

### Tools and Technologies

Python, Pandas, NumPy, Matplotlib, Seaborn, SciPy, Statsmodels and Scikit-learn.

### Research Approach

Data cleaning → Exploratory analysis → Statistical testing → Customer segmentation → Predictive modelling → Retention strategy.

**Important:** This project uses observational data. Its findings identify associations and potential business opportunities, not proven causal effects.

In [ ]:
!pip -q install ucimlrepo openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

print("SIGNAL environment ready.")


SIGNAL environment ready.


In [ ]:

import requests

url = (
    "https://raw.githubusercontent.com/"
    "mimansha98/Tableau_Dashboards/"
    "master/sample-superstore.csv"
)

response = requests.get(url, timeout=60)

print("HTTP STATUS:", response.status_code)
print("CONTENT TYPE:", response.headers.get("Content-Type"))
print("FILE SIZE:", len(response.content), "bytes")

print("\nFIRST 1000 CHARACTERS:")
print(response.content[:1000].decode("cp1252", errors="replace"))

print("\nFIRST 10 LINES:")

text = response.content.decode("cp1252", errors="replace")

for i, line in enumerate(text.splitlines()[:10]):
    print(i, repr(line))


HTTP STATUS: 200
CONTENT TYPE: text/plain; charset=utf-16
FILE SIZE: 4154538 bytes

FIRST 1000 CHARACTERS:
ÿþP r o f i t   R a t i o 	 O N E 	 C a t e g o r y 	 C i t y 	 C o u n t r y 	 C u s t o m e r   N a m e 	 D i s c o u n t 	 N u m b e r   o f   R e c o r d s 	 O r d e r   D a t e 	 O r d e r   I D 	 P o s t a l   C o d e 	 M a n u f a c t u r e r 	 P r o d u c t   N a m e 	 P r o f i t 	 Q u a n t i t y 	 R e g i o n 	 S a l e s 	 S e g m e n t 	 S h i p   D a t e 	 S h i p   M o d e 	 S t a t e 	 S u b - C a t e g o r y 
 1 6 % 	 1 	 F u r n i t u r e 	 H e n d e r s o n 	 U n i t e d   S t a t e s 	 C l a i r e   G u t e 	 0 % 	 1 	 1 1 / 8 / 2 0 1 7 	 C A - 2 0 1 7 - 1 5 2 1 5 6 	 4 2 4 2 0 	 B u s h 	 B u s h   S o m e r s e t   C o l l e c t i o n   B o o k c a s e 	 $ 4 2 	 2 	 S o u t h 	 $ 2 6 2 	 C o n s u m e r 	 1 1 / 1 1 / 2 0 1 7 	 S e c o n d   C l a s s 	 K e n t u c k y 	 B o o k c a s e s 
 3 0 % 	 1 	 F u r n i t u r e 	 H e n d e r s o n 	 U n i t e d   S t a

In [ ]:
from io import BytesIO

url = (
    "https://raw.githubusercontent.com/"
    "mimansha98/Tableau_Dashboards/"
    "master/sample-superstore.csv"
)

response = requests.get(url, timeout=60)
response.raise_for_status()

# Correct encoding and delimiter
df_superstore = pd.read_csv(
    BytesIO(response.content),
    encoding="utf-16",
    sep="\t"
)

# Preserve untouched source
df_superstore_raw = df_superstore.copy()

print("PROJECT SIGNAL — DATA LOADED")
print("Shape:", df_superstore.shape)
print("Columns:", df_superstore.columns.tolist())

display(df_superstore.head())


PROJECT SIGNAL — DATA LOADED
Shape: (9994, 22)
Columns: ['Profit Ratio', 'ONE', 'Category', 'City', 'Country', 'Customer Name', 'Discount', 'Number of Records', 'Order Date', 'Order ID', 'Postal Code', 'Manufacturer', 'Product Name', 'Profit', 'Quantity', 'Region', 'Sales', 'Segment', 'Ship Date', 'Ship Mode', 'State', 'Sub-Category']


,Profit Ratio,ONE,Category,City,Country,Customer Name,Discount,Number of Records,Order Date,Order ID,...,Product Name,Profit,Quantity,Region,Sales,Segment,Ship Date,Ship Mode,State,Sub-Category
0,16%,1,Furniture,Henderson,United States,Claire Gute,0%,1,11/8/2017,CA-2017-152156,...,Bush Somerset Collection Bookcase,$42,2,South,$262,Consumer,11/11/2017,Second Class,Kentucky,Bookcases
1,30%,1,Furniture,Henderson,United States,Claire Gute,0%,1,11/8/2017,CA-2017-152156,...,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",$220,3,South,$732,Consumer,11/11/2017,Second Class,Kentucky,Chairs
2,47%,1,Office Supplies,Los Angeles,United States,Darrin Van Huff,0%,1,6/12/2017,CA-2017-138688,...,Self-Adhesive Address Labels for Typewriters b...,$7,2,West,$15,Corporate,6/16/2017,Second Class,California,Labels
3,-40%,1,Furniture,Fort Lauderdale,United States,Sean O'Donnell,45%,1,10/11/2016,US-2016-108966,...,Bretford CR4500 Series Slim Rectangular Table,-$383,5,South,$958,Consumer,10/18/2016,Standard Class,Florida,Tables
4,11%,1,Office Supplies,Fort Lauderdale,United States,Sean O'Donnell,20%,1,10/11/2016,US-2016-108966,...,Eldon Fold 'N Roll Cart System,$3,2,South,$22,Consumer,10/18/2016,Standard Class,Florida,Storage


In [ ]:

required_columns = [
    "Customer Name",
    "Order ID",
    "Order Date",
    "Sales",
    "Discount",
    "Profit",
    "Quantity",
    "Product Name"
]

for col in required_columns:
    print(
        f"{col}:",
        "AVAILABLE" if col in df_superstore.columns
        else "MISSING"
    )


Customer Name: AVAILABLE
Order ID: AVAILABLE
Order Date: AVAILABLE
Sales: AVAILABLE
Discount: AVAILABLE
Profit: AVAILABLE
Quantity: AVAILABLE
Product Name: AVAILABLE


## Data audit

In [ ]:
# Keeping the original dataset untouched
audit_df = df_superstore_raw.copy()

print("=" * 60)
print("PROJECT SIGNAL — INITIAL DATA AUDIT")
print("=" * 60)

#    Dataset dimensions
print("\n1. DATASET SIZE")
print("Rows:", audit_df.shape[0])
print("Columns:", audit_df.shape[1])

#    Data types
print("\n2. COLUMN DATA TYPES")
display(audit_df.dtypes.to_frame("Data Type"))

#    Missing values
print("\n3. MISSING VALUES")
missing = audit_df.isna().sum()
missing_report = pd.DataFrame({
    "Missing Values": missing,
    "Missing Percentage": (missing / len(audit_df) * 100).round(2)
})
display(missing_report[missing_report["Missing Values"] > 0])

if missing.sum() == 0:
    print("No missing values found.")

#    Duplicate records
print("\n4. DUPLICATE RECORDS")
print("Exact duplicate rows:", audit_df.duplicated().sum())

#    Order and customer structure
print("\n5. ORDER AND CUSTOMER COUNTS")
print("Unique orders:", audit_df["Order ID"].nunique())
print("Unique customer names:", audit_df["Customer Name"].nunique())
print("Unique products:", audit_df["Product Name"].nunique())

#     Customer name consistency
orders_per_customer = (
    audit_df.groupby("Customer Name")["Order ID"].nunique()
)

print("\n6. CUSTOMER ORDER DISTRIBUTION")
print("Customers with one order:", (orders_per_customer == 1).sum())
print("Customers with multiple orders:", (orders_per_customer > 1).sum())
print("Maximum orders associated with one customer:",
      orders_per_customer.max())

#   Date coverage
order_dates = pd.to_datetime(
    audit_df["Order Date"],
    format="%m/%d/%Y",
    errors="coerce"
)

print("\n7. ORDER DATE COVERAGE")
print("Earliest date:", order_dates.min())
print("Latest date:", order_dates.max())
print("Unparseable order dates:", order_dates.isna().sum())

#   Tableau-specific columns
print("\n8. SPECIAL COLUMNS")
for col in ["ONE", "Number of Records", "Profit Ratio"]:
    if col in audit_df.columns:
        print(f"\n{col}")
        print("Unique values:", audit_df[col].nunique())
        print(audit_df[col].value_counts(dropna=False).head(10))

#  Check whether customer names map to multiple segments
segments_per_customer = (
    audit_df.groupby("Customer Name")["Segment"].nunique()
)

print("\n9. CUSTOMER-SEGMENT CONSISTENCY")
print(
    "Names associated with multiple segments:",
    (segments_per_customer > 1).sum()
)

print("\n" + "=" * 60)
print("INITIAL AUDIT COMPLETE")
print("=" * 60)


PROJECT SIGNAL — INITIAL DATA AUDIT

1. DATASET SIZE
Rows: 9994
Columns: 22

2. COLUMN DATA TYPES


,Data Type
Profit Ratio,object
ONE,int64
Category,object
City,object
Country,object
Customer Name,object
Discount,object
Number of Records,int64
Order Date,object
Order ID,object



3. MISSING VALUES


,Missing Values,Missing Percentage
Postal Code,11,0.11



4. DUPLICATE RECORDS
Exact duplicate rows: 1

5. ORDER AND CUSTOMER COUNTS
Unique orders: 5009
Unique customer names: 793
Unique products: 1850

6. CUSTOMER ORDER DISTRIBUTION
Customers with one order: 12
Customers with multiple orders: 781
Maximum orders associated with one customer: 17

7. ORDER DATE COVERAGE
Earliest date: 2015-01-03 00:00:00
Latest date: 2018-12-30 00:00:00
Unparseable order dates: 0

8. SPECIAL COLUMNS

ONE
Unique values: 1
ONE
1    9994
Name: count, dtype: int64

Number of Records
Unique values: 1
Number of Records
1    9994
Name: count, dtype: int64

Profit Ratio
Unique values: 161
Profit Ratio
48%    509
35%    443
26%    364
34%    360
47%    353
36%    338
33%    320
49%    315
29%    303
28%    299
Name: count, dtype: int64

9. CUSTOMER-SEGMENT CONSISTENCY
Names associated with multiple segments: 0

INITIAL AUDIT COMPLETE


## the duplicate and missing values

In [ ]:
#  Finding exact duplicate rows
duplicate_rows = audit_df[
    audit_df.duplicated(keep=False)
].sort_values(["Order ID", "Product Name"])

print("DUPLICATE RECORDS")
print("Rows involved:", len(duplicate_rows))

display(duplicate_rows)

#  Inspecting missing values across every column
print("\nMISSING VALUE AUDIT")

missing_report = pd.DataFrame({
    "Missing Count": audit_df.isna().sum(),
    "Missing Percentage": (
        audit_df.isna().mean() * 100
    ).round(2)
})

display(missing_report)

#  Inspecting possible order-line duplicates
print("\nORDER-LINE INVESTIGATION")

line_columns = [
    "Order ID",
    "Product Name",
    "Quantity",
    "Sales",
    "Profit"
]

print(
    "Duplicate combinations of order and product:",
    audit_df.duplicated(
        subset=line_columns,
        keep=False
    ).sum()
)

#  Inspecting data types
print("\nDATA TYPES")
display(audit_df.dtypes.to_frame("Data Type"))


DUPLICATE RECORDS
Rows involved: 2


,Profit Ratio,ONE,Category,City,Country,Customer Name,Discount,Number of Records,Order Date,Order ID,...,Product Name,Profit,Quantity,Region,Sales,Segment,Ship Date,Ship Mode,State,Sub-Category
8606,-4%,1,Furniture,Columbus,United States,Laurel Beltran,30%,1,4/23/2015,US-2015-150119,...,Global Leather Highback Executive Chair with P...,-$12,2,East,$281,Home Office,4/27/2015,Standard Class,Ohio,Chairs
8607,-4%,1,Furniture,Columbus,United States,Laurel Beltran,30%,1,4/23/2015,US-2015-150119,...,Global Leather Highback Executive Chair with P...,-$12,2,East,$281,Home Office,4/27/2015,Standard Class,Ohio,Chairs



MISSING VALUE AUDIT


,Missing Count,Missing Percentage
Profit Ratio,0,0.00
ONE,0,0.00
Category,0,0.00
City,0,0.00
Country,0,0.00
Customer Name,0,0.00
Discount,0,0.00
Number of Records,0,0.00
Order Date,0,0.00
Order ID,0,0.00



ORDER-LINE INVESTIGATION
Duplicate combinations of order and product: 2

DATA TYPES


,Data Type
Profit Ratio,object
ONE,int64
Category,object
City,object
Country,object
Customer Name,object
Discount,object
Number of Records,int64
Order Date,object
Order ID,object


# the cleaned dataset

In [ ]:
# Starting from the untouched original dataset
df_clean = df_superstore_raw.copy()

print("PROJECT SIGNAL — DATA CLEANING")
print("-" * 50)

#  Removing exact duplicate rows
rows_before = len(df_clean)

df_clean = df_clean.drop_duplicates(keep="first").copy()

rows_after = len(df_clean)

print("\n1. DUPLICATE REMOVAL")
print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Duplicates removed:", rows_before - rows_after)

# STEP 2: Convert currency columns to numeric
currency_columns = ["Sales", "Profit"]

for col in currency_columns:
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .astype(float)
    )

#  Converting percentage columns to decimal values
percentage_columns = ["Discount", "Profit Ratio"]

for col in percentage_columns:
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.strip()
        .astype(float) / 100
    )

#   Converting dates
date_columns = ["Order Date", "Ship Date"]

for col in date_columns:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        format="%m/%d/%Y",
        errors="coerce"
    )

#  Converting postal code to nullable integer
df_clean["Postal Code"] = df_clean["Postal Code"].astype("Int64")

print("\n2. DATA TYPE CONVERSION")
display(df_clean.dtypes.to_frame("New Data Type"))

#  Verifying missing values
print("\n3. MISSING VALUES AFTER CLEANING")
display(
    df_clean.isna().sum()
    .to_frame("Missing Values")
    .query("`Missing Values` > 0")
)

#  Verifying conversions
print("\n4. CONVERSION CHECK")
display(
    df_clean[
        ["Sales", "Profit", "Discount", "Profit Ratio",
         "Order Date", "Ship Date"]
    ].head()
)

#  Verifying remaining duplicates
print("\n5. FINAL DUPLICATE CHECK")
print("Exact duplicates remaining:", df_clean.duplicated().sum())

print("\nFINAL CLEAN DATASET SHAPE:", df_clean.shape)
print("\nCLEANING COMPLETE")


PROJECT SIGNAL — DATA CLEANING
--------------------------------------------------

1. DUPLICATE REMOVAL
Rows before: 9994
Rows after: 9993
Duplicates removed: 1

2. DATA TYPE CONVERSION


,New Data Type
Profit Ratio,float64
ONE,int64
Category,object
City,object
Country,object
Customer Name,object
Discount,float64
Number of Records,int64
Order Date,datetime64[ns]
Order ID,object



3. MISSING VALUES AFTER CLEANING


,Missing Values
Postal Code,11



4. CONVERSION CHECK


,Sales,Profit,Discount,Profit Ratio,Order Date,Ship Date
0,262.0,42.0,0.00,0.16,2017-11-08,2017-11-11
1,732.0,220.0,0.00,0.30,2017-11-08,2017-11-11
2,15.0,7.0,0.00,0.47,2017-06-12,2017-06-16
3,958.0,-383.0,0.45,-0.40,2016-10-11,2016-10-18
4,22.0,3.0,0.20,0.11,2016-10-11,2016-10-18



5. FINAL DUPLICATE CHECK
Exact duplicates remaining: 0

FINAL CLEAN DATASET SHAPE: (9993, 22)

CLEANING COMPLETE


In [ ]:
print("PROJECT SIGNAL — FINANCIAL VALIDATION")
print("=" * 55)

#  Independently calculate profit ratio
df_validation = df_clean.copy()

df_validation["Calculated Profit Ratio"] = np.where(
    df_validation["Sales"] != 0,
    df_validation["Profit"] / df_validation["Sales"],
    np.nan
)

#  Compare rounded ratios
df_validation["Calculated Ratio Rounded"] = (
    df_validation["Calculated Profit Ratio"].round(2)
)

df_validation["Ratio Difference"] = (
    df_validation["Profit Ratio"]
    - df_validation["Calculated Ratio Rounded"]
).abs()

# Identifying mismatches
mismatches = df_validation[
    df_validation["Ratio Difference"] > 0.01
]

print("\n1. PROFIT RATIO CONSISTENCY")
print("Rows checked:", df_validation["Calculated Profit Ratio"].notna().sum())
print("Mismatched rows:", len(mismatches))

print("\nMismatch percentage:",
      round(len(mismatches) / len(df_validation) * 100, 3), "%")

if len(mismatches) > 0:
    display(
        mismatches[
            ["Sales", "Profit", "Profit Ratio",
             "Calculated Profit Ratio", "Ratio Difference"]
        ].head(10)
    )

#  Checking financial ranges
print("\n2. FINANCIAL VALUE CHECKS")

for col in ["Sales", "Profit", "Discount", "Profit Ratio"]:
    print(f"\n{col}")
    print("Minimum:", df_clean[col].min())
    print("Maximum:", df_clean[col].max())
    print("Negative values:", (df_clean[col] < 0).sum())

#   Checking order and shipping dates
print("\n3. DATE CONSISTENCY")
print(
    "Orders shipped before order date:",
    (df_clean["Ship Date"] < df_clean["Order Date"]).sum()
)

#   Checking quantity
print("\n4. QUANTITY CHECK")
print("Zero quantity:", (df_clean["Quantity"] == 0).sum())
print("Negative quantity:", (df_clean["Quantity"] < 0).sum())

print("\n" + "=" * 55)
print("FINANCIAL VALIDATION COMPLETE")


PROJECT SIGNAL — FINANCIAL VALIDATION

1. PROFIT RATIO CONSISTENCY
Rows checked: 9992
Mismatched rows: 4046

Mismatch percentage: 40.488 %


,Sales,Profit,Profit Ratio,Calculated Profit Ratio,Ratio Difference
4,22.0,3.0,0.11,0.136364,0.03
6,7.0,2.0,0.27,0.285714,0.02
8,19.0,6.0,0.31,0.315789,0.01
12,16.0,5.0,0.35,0.312500,0.04
15,3.0,-4.0,-1.50,-1.333333,0.17
18,9.0,2.0,0.29,0.222222,0.07
20,23.0,7.0,0.33,0.304348,0.03
22,60.0,16.0,0.26,0.266667,0.01
25,12.0,4.0,0.36,0.333333,0.03
28,10.0,-7.0,-0.73,-0.700000,0.03



2. FINANCIAL VALUE CHECKS

Sales
Minimum: 0.0
Maximum: 22638.0
Negative values: 0

Profit
Minimum: -6600.0
Maximum: 8400.0
Negative values: 1864

Discount
Minimum: 0.0
Maximum: 0.8
Negative values: 0

Profit Ratio
Minimum: -2.75
Maximum: 0.5
Negative values: 1870

3. DATE CONSISTENCY
Orders shipped before order date: 0

4. QUANTITY CHECK
Zero quantity: 0
Negative quantity: 0

FINANCIAL VALIDATION COMPLETE


# — FINANCIAL ANOMALY INVESTIGATION

In [ ]:
#  Inspecting records with zero sales
zero_sales = df_clean[df_clean["Sales"] == 0]

print("1. ZERO-SALES RECORDS")
print("Number of records:", len(zero_sales))

display(
    zero_sales[
        ["Order ID", "Customer Name", "Product Name",
         "Sales", "Profit", "Discount", "Profit Ratio"]
    ]
)

#  Comparing supplied ratio with ratio calculated from displayed values
ratio_check = df_clean[df_clean["Sales"] > 0].copy()

ratio_check["Calculated Margin"] = (
    ratio_check["Profit"] / ratio_check["Sales"]
)

ratio_check["Profit Implied By Source Ratio"] = (
    ratio_check["Sales"] * ratio_check["Profit Ratio"]
)

ratio_check["Profit Difference"] = (
    ratio_check["Profit"] -
    ratio_check["Profit Implied By Source Ratio"]
)

print("\n2. PROFIT DIFFERENCE SUMMARY")
display(
    ratio_check["Profit Difference"].describe().to_frame(
        "Difference in dollars"
    )
)

#  Inspecting largest differences
print("\n3. TEN LARGEST ABSOLUTE DIFFERENCES")

largest_differences = (
    ratio_check.assign(
        Absolute_Difference=lambda x: x["Profit Difference"].abs()
    )
    .sort_values("Absolute_Difference", ascending=False)
)

display(
    largest_differences[
        ["Sales", "Profit", "Profit Ratio",
         "Calculated Margin", "Profit Difference"]
    ].head(10)
)

#  Checking whether source ratio has whole-percentage precision
print("\n4. SOURCE RATIO PRECISION")
print(
    "Unique decimal places in supplied ratios:",
    sorted(
        ratio_check["Profit Ratio"]
        .dropna()
        .map(lambda x: len(str(x).split(".")[-1]) if "." in str(x) else 0)
        .unique()
    )
)

print("\nINVESTIGATION COMPLETE")


1. ZERO-SALES RECORDS
Number of records: 1


,Order ID,Customer Name,Product Name,Sales,Profit,Discount,Profit Ratio
3456,US-2018-102288,Zuschuss Carroll,Hoover Replacement Belt for Commercial Guardsm...,0.0,-1.0,0.8,-2.5



2. PROFIT DIFFERENCE SUMMARY


,Difference in dollars
count,9992.000000
mean,-0.048625
std,1.228547
min,-25.920000
25%,-0.300000
50%,0.000000
75%,0.260000
max,28.000000



3. TEN LARGEST ABSOLUTE DIFFERENCES


,Sales,Profit,Profit Ratio,Calculated Margin,Profit Difference
5379,8400.0,1120.0,0.13,0.133333,28.00
165,8160.0,-1360.0,-0.17,-0.166667,27.20
7830,5084.0,1906.0,0.38,0.374902,-25.92
7938,4800.0,360.0,0.07,0.075000,24.00
2845,4664.0,-1049.0,-0.23,-0.224914,23.72
353,4355.0,1415.0,0.33,0.324914,-22.15
5094,4368.0,328.0,0.08,0.075092,-21.44
4688,3266.0,1062.0,0.33,0.325168,-15.78
2167,3050.0,1144.0,0.38,0.375082,-15.00
5398,4500.0,-6600.0,-1.47,-1.466667,15.00



4. SOURCE RATIO PRECISION
Unique decimal places in supplied ratios: [np.int64(1), np.int64(2)]

INVESTIGATION COMPLETE


# — ANALYTICAL TABLE CONSTRUCTION


In [ ]:
# CHECKING ORDER CONSISTENCY

print("1. ORDER CONSISTENCY CHECK")

customers_per_order = (
    df_clean.groupby("Order ID")["Customer Name"].nunique()
)

dates_per_order = (
    df_clean.groupby("Order ID")["Order Date"].nunique()
)

print("Orders associated with multiple customer names:",
      (customers_per_order > 1).sum())

print("Orders associated with multiple order dates:",
      (dates_per_order > 1).sum())

#  Building Order Level Table


df_orders = (
    df_clean.groupby("Order ID")
    .agg(
        Customer_Name=("Customer Name", "first"),
        Order_Date=("Order Date", "min"),
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Total_Quantity=("Quantity", "sum"),
        Product_Lines=("Product Name", "size"),
        Unique_Products=("Product Name", "nunique"),
        Mean_Line_Discount=("Discount", "mean"),
        Maximum_Line_Discount=("Discount", "max"),
        Segment=("Segment", "first")
    )
    .reset_index()
)

#  Calculating order-level profit margin
df_orders["Profit_Margin"] = np.where(
    df_orders["Sales"] > 0,
    df_orders["Profit"] / df_orders["Sales"],
    np.nan
)

print("\n2. ORDER-LEVEL TABLE")
print("Shape:", df_orders.shape)
display(df_orders.head())


#  Building customer level table


df_customers = (
    df_orders.groupby("Customer_Name")
    .agg(
        Order_Count=("Order ID", "nunique"),
        First_Order=("Order_Date", "min"),
        Last_Order=("Order_Date", "max"),
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Total_Quantity=("Total_Quantity", "sum"),
        Average_Order_Value=("Sales", "mean"),
        Average_Order_Profit=("Profit", "mean"),
        Mean_Order_Line_Discount=("Mean_Line_Discount", "mean"),
        Segment=("Segment", "first")
    )
    .reset_index()
)

#  Adding distinct products purchased by each customer
customer_product_counts = (
    df_clean.groupby("Customer Name")["Product Name"]
    .nunique()
    .rename("Distinct_Products")
    .reset_index()
)

df_customers = df_customers.merge(
    customer_product_counts,
    left_on="Customer_Name",
    right_on="Customer Name",
    how="left"
)

df_customers.drop(columns=["Customer Name"], inplace=True)

#  Customer retention classification
df_customers["Repeat_Customer"] = (
    df_customers["Order_Count"] >= 2
)

# Customer-level profit margin
df_customers["Profit_Margin"] = np.where(
    df_customers["Total_Sales"] > 0,
    df_customers["Total_Profit"] / df_customers["Total_Sales"],
    np.nan
)


# Verifying both tables

print("\n3. CUSTOMER-LEVEL TABLE")
print("Shape:", df_customers.shape)
display(df_customers.head())

print("\n4. RETENTION COUNTS")
print(df_customers["Repeat_Customer"].value_counts())

print("\n5. FINAL STRUCTURAL CHECKS")
print("Original cleaned transaction rows:", len(df_clean))
print("Order-level records:", len(df_orders))
print("Customer-level records:", len(df_customers))
print("Total order sales:", round(df_orders["Sales"].sum(), 2))
print("Total transaction sales:", round(df_clean["Sales"].sum(), 2))
print("Total order profit:", round(df_orders["Profit"].sum(), 2))
print("Total transaction profit:", round(df_clean["Profit"].sum(), 2))

print("\nANALYTICAL TABLES CREATED")


1. ORDER CONSISTENCY CHECK
Orders associated with multiple customer names: 0
Orders associated with multiple order dates: 0

2. ORDER-LEVEL TABLE
Shape: (5009, 12)


,Order ID,Customer_Name,Order_Date,Sales,Profit,Total_Quantity,Product_Lines,Unique_Products,Mean_Line_Discount,Maximum_Line_Discount,Segment,Profit_Margin
0,CA-2015-100006,Dennis Kane,2015-09-07,378.0,110.0,3,1,1,0.0,0.0,Consumer,0.291005
1,CA-2015-100090,Ed Braxton,2015-07-08,699.0,-19.0,9,2,2,0.2,0.2,Corporate,-0.027182
2,CA-2015-100293,Neil Französisch,2015-03-14,91.0,32.0,6,1,1,0.2,0.2,Home Office,0.351648
3,CA-2015-100328,Jasper Cacioppo,2015-01-28,4.0,1.0,1,1,1,0.2,0.2,Consumer,0.250000
4,CA-2015-100363,Jim Mitchum,2015-04-08,21.0,8.0,5,2,2,0.2,0.2,Corporate,0.380952



3. CUSTOMER-LEVEL TABLE
Shape: (793, 14)


,Customer_Name,Order_Count,First_Order,Last_Order,Total_Sales,Total_Profit,Total_Quantity,Average_Order_Value,Average_Order_Profit,Mean_Order_Line_Discount,Segment,Distinct_Products,Repeat_Customer,Profit_Margin
0,Aaron Bergman,3,2015-02-18,2017-11-10,887.0,129.0,13,295.666667,43.000000,0.088889,Consumer,6,True,0.145434
1,Aaron Hawkins,7,2015-04-22,2018-12-18,1744.0,364.0,54,249.142857,52.000000,0.100000,Corporate,11,True,0.208716
2,Aaron Smayling,7,2015-07-27,2018-10-03,3051.0,-254.0,48,435.857143,-36.285714,0.283333,Corporate,10,True,-0.083251
3,Adam Bellavance,8,2016-09-18,2018-11-06,7757.0,2055.0,56,969.625000,256.875000,0.043333,Home Office,17,True,0.264922
4,Adam Hart,10,2015-11-16,2018-11-26,3251.0,283.0,75,325.100000,28.300000,0.135000,Corporate,20,True,0.087050



4. RETENTION COUNTS
Repeat_Customer
True     781
False     12
Name: count, dtype: int64

5. FINAL STRUCTURAL CHECKS
Original cleaned transaction rows: 9993
Order-level records: 5009
Customer-level records: 793
Total order sales: 2297073.0
Total transaction sales: 2297073.0
Total order profit: 286359.0
Total transaction profit: 286359.0

ANALYTICAL TABLES CREATED


## Customer behaviour exploration

In [ ]:
print("=" * 60)
print("CUSTOMER PURCHASING BEHAVIOUR")
print("=" * 60)

#  Ordering frequency distribution
frequency_distribution = (
    df_customers["Order_Count"]
    .value_counts()
    .sort_index()
    .rename_axis("Number of Orders")
    .reset_index(name="Number of Customers")
)

print("\n1. ORDER FREQUENCY DISTRIBUTION")
display(frequency_distribution)

#   Customer purchasing statistics
print("\n2. CUSTOMER ORDER STATISTICS")

print("Total customers:", len(df_customers))
print("Average orders per customer:",
      round(df_customers["Order_Count"].mean(), 2))
print("Median orders per customer:",
      df_customers["Order_Count"].median())
print("Minimum orders:", df_customers["Order_Count"].min())
print("Maximum orders:", df_customers["Order_Count"].max())

#  Calculating time between consecutive orders
sorted_orders = df_orders.sort_values(
    ["Customer_Name", "Order_Date"]
).copy()

sorted_orders["Days_Between_Orders"] = (
    sorted_orders.groupby("Customer_Name")["Order_Date"]
    .diff()
    .dt.days
)

#  Summarising purchase intervals per customer
purchase_intervals = (
    sorted_orders.groupby("Customer_Name")
    ["Days_Between_Orders"]
    .agg(
        Average_Days_Between_Orders="mean",
        Median_Days_Between_Orders="median",
        Number_of_Intervals="count"
    )
    .reset_index()
)

#    Attach interval metrics to customer table
df_customers = df_customers.merge(
    purchase_intervals,
    on="Customer_Name",
    how="left"
)

#   Overall interval statistics
print("\n3. TIME BETWEEN PURCHASES")

print(
    "Customers with at least one measurable interval:",
    df_customers["Number_of_Intervals"].gt(0).sum()
)

print(
    "Median customer-level purchase interval:",
    round(
        df_customers["Median_Days_Between_Orders"].median(),
        2
    ),
    "days"
)

print(
    "Median of all individual purchase intervals:",
    round(
        sorted_orders["Days_Between_Orders"].median(),
        2
    ),
    "days"
)

#  Highest-frequency customers
print("\n4. HIGHEST ORDER-FREQUENCY CUSTOMERS")

display(
    df_customers.sort_values(
        "Order_Count",
        ascending=False
    )[
        ["Customer_Name", "Order_Count",
         "Total_Sales", "Total_Profit",
         "Profit_Margin", "Segment"]
    ].head(10)
)

print("\nCUSTOMER BEHAVIOUR EDA COMPLETE")


CUSTOMER PURCHASING BEHAVIOUR

1. ORDER FREQUENCY DISTRIBUTION


,Number of Orders,Number of Customers
0,1,12
1,2,34
2,3,53
3,4,96
4,5,134
5,6,107
6,7,116
7,8,82
8,9,71
9,10,39



2. CUSTOMER ORDER STATISTICS
Total customers: 793
Average orders per customer: 6.32
Median orders per customer: 6.0
Minimum orders: 1
Maximum orders: 17

3. TIME BETWEEN PURCHASES
Customers with at least one measurable interval: 781
Median customer-level purchase interval: 145.0 days
Median of all individual purchase intervals: 128.0 days

4. HIGHEST ORDER-FREQUENCY CUSTOMERS


,Customer_Name,Order_Count,Total_Sales,Total_Profit,Profit_Margin,Segment
259,Emily Phan,17,5481.0,146.0,0.026637,Consumer
663,Sally Hughsby,13,3408.0,558.0,0.163732,Corporate
574,Noel Staavos,13,2966.0,-236.0,-0.079568,Corporate
586,Patrick Gardner,13,3088.0,135.0,0.043718,Consumer
791,Zuschuss Carroll,13,8028.0,-1032.0,-0.128550,Consumer
381,Joel Eaton,13,6762.0,225.0,0.033274,Consumer
267,Erin Ashbrook,13,2846.0,-51.0,-0.017920,Corporate
145,Chloris Kastensmidt,13,3154.0,143.0,0.045339,Consumer
728,Suzanne McNair,12,5563.0,580.0,0.104260,Corporate
629,Rick Bensley,12,4714.0,640.0,0.135766,Home Office



CUSTOMER BEHAVIOUR EDA COMPLETE


## does frequency translate into profitability?

In [ ]:
analysis_df = df_customers.copy()

#  Creating  purchasing-frequency groups
bins = [0, 1, 3, 6, 9, float("inf")]

labels = [
    "1 order",
    "2-3 orders",
    "4-6 orders",
    "7-9 orders",
    "10+ orders"
]

analysis_df["Frequency_Group"] = pd.cut(
    analysis_df["Order_Count"],
    bins=bins,
    labels=labels
)

#  Aggregating customer economics by frequency group
frequency_economics = (
    analysis_df.groupby("Frequency_Group", observed=False)
    .agg(
        Customer_Count=("Customer_Name", "count"),
        Average_Orders=("Order_Count", "mean"),
        Median_Orders=("Order_Count", "median"),
        Average_Customer_Sales=("Total_Sales", "mean"),
        Median_Customer_Sales=("Total_Sales", "median"),
        Average_Customer_Profit=("Total_Profit", "mean"),
        Median_Customer_Profit=("Total_Profit", "median"),
        Total_Sales=("Total_Sales", "sum"),
        Total_Profit=("Total_Profit", "sum"),
        Average_Order_Value=("Average_Order_Value", "mean")
    )
    .reset_index()
)

#   Calculating group-level profit margin correctly
frequency_economics["Aggregate_Profit_Margin"] = np.where(
    frequency_economics["Total_Sales"] > 0,
    frequency_economics["Total_Profit"] /
    frequency_economics["Total_Sales"],
    np.nan
)

#    Counting customers who generated a net loss
loss_counts = (
    analysis_df.assign(
        Loss_Making=analysis_df["Total_Profit"] < 0
    )
    .groupby("Frequency_Group", observed=False)
    .agg(
        Loss_Making_Customers=("Loss_Making", "sum")
    )
    .reset_index()
)

frequency_economics = frequency_economics.merge(
    loss_counts,
    on="Frequency_Group"
)

frequency_economics["Loss_Making_Percentage"] = (
    frequency_economics["Loss_Making_Customers"] /
    frequency_economics["Customer_Count"] * 100
).round(2)

#   Displaying findings
print("CUSTOMER ECONOMICS BY PURCHASING FREQUENCY")
display(frequency_economics.round(2))

#   Checking overall customer profitability
print("\nOVERALL CUSTOMER PROFITABILITY")

print(
    "Profitable customers:",
    (analysis_df["Total_Profit"] > 0).sum()
)

print(
    "Loss-making customers:",
    (analysis_df["Total_Profit"] < 0).sum()
)

print(
    "Zero-profit customers:",
    (analysis_df["Total_Profit"] == 0).sum()
)

print("\nCUSTOMER VALUE ANALYSIS COMPLETE")


CUSTOMER ECONOMICS BY PURCHASING FREQUENCY


,Frequency_Group,Customer_Count,Average_Orders,Median_Orders,Average_Customer_Sales,Median_Customer_Sales,Average_Customer_Profit,Median_Customer_Profit,Total_Sales,Total_Profit,Average_Order_Value,Aggregate_Profit_Margin,Loss_Making_Customers,Loss_Making_Percentage
0,1 order,12,1.00,1.0,430.50,129.5,64.00,20.5,5166.0,768.0,430.50,0.15,2,16.67
1,2-3 orders,87,2.61,3.0,1137.13,759.0,53.11,61.0,98930.0,4621.0,429.85,0.05,22,25.29
2,4-6 orders,337,5.03,5.0,2403.61,1759.0,306.14,186.0,810015.0,103169.0,477.16,0.13,76,22.55
3,7-9 orders,269,7.83,8.0,3489.76,3051.0,440.81,338.0,938746.0,118578.0,449.91,0.13,44,16.36
4,10+ orders,88,10.99,11.0,5047.91,4050.5,672.99,527.0,444216.0,59223.0,460.15,0.13,11,12.50



OVERALL CUSTOMER PROFITABILITY
Profitable customers: 636
Loss-making customers: 155
Zero-profit customers: 2

CUSTOMER VALUE ANALYSIS COMPLETE


 # Do higher discounts increase sales while reducing profitability?

In [ ]:

discount_df = df_clean.copy()

#  Creating discount-level groups
discount_economics = (
    discount_df.groupby("Discount")
    .agg(
        Transaction_Lines=("Order ID", "size"),
        Unique_Orders=("Order ID", "nunique"),
        Unique_Customers=("Customer Name", "nunique"),
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Average_Line_Sales=("Sales", "mean"),
        Average_Line_Profit=("Profit", "mean"),
        Loss_Making_Lines=("Profit", lambda x: (x < 0).sum())
    )
    .reset_index()
)

#  Calculating aggregate profit margin
discount_economics["Profit_Margin"] = np.where(
    discount_economics["Total_Sales"] > 0,
    discount_economics["Total_Profit"] /
    discount_economics["Total_Sales"],
    np.nan
)

#  Calculating loss-making percentage
discount_economics["Loss_Making_Percentage"] = (
    discount_economics["Loss_Making_Lines"] /
    discount_economics["Transaction_Lines"] * 100
)

#  Formating discount as percentage for readability
discount_economics["Discount_Percentage"] = (
    discount_economics["Discount"] * 100
)

discount_economics = discount_economics.sort_values(
    "Discount"
)

print("1. DISCOUNT ECONOMICS BY DISCOUNT LEVEL")
display(discount_economics.round(3))

#  Overall relationship checks
print("\n2. DISCOUNT AND PROFITABILITY")

print(
    "Correlation: discount vs transaction-line profit:",
    round(discount_df["Discount"].corr(discount_df["Profit"]), 3)
)

print(
    "Correlation: discount vs transaction-line sales:",
    round(discount_df["Discount"].corr(discount_df["Sales"]), 3)
)

print(
    "Correlation: discount vs transaction-line profit margin:",
    round(
        discount_df.loc[
            discount_df["Sales"] > 0, "Discount"
        ].corr(
            (
                discount_df.loc[
                    discount_df["Sales"] > 0, "Profit"
                ] /
                discount_df.loc[
                    discount_df["Sales"] > 0, "Sales"
                ]
            )
        ),
        3
    )
)

print("\nDISCOUNT ECONOMICS ANALYSIS COMPLETE")


1. DISCOUNT ECONOMICS BY DISCOUNT LEVEL


,Discount,Transaction_Lines,Unique_Orders,Unique_Customers,Total_Sales,Total_Profit,Average_Line_Sales,Average_Line_Profit,Loss_Making_Lines,Profit_Margin,Loss_Making_Percentage,Discount_Percentage
0,0.00,4798,2644,759,1088020.0,320946.0,226.765,66.892,0,0.295,0.000,0.0
1,0.10,94,89,85,54376.0,9025.0,578.468,96.011,4,0.166,4.255,10.0
2,0.15,52,51,49,27560.0,1416.0,530.000,27.231,17,0.051,32.692,15.0
3,0.20,3657,2407,758,764636.0,90344.0,209.088,24.704,496,0.118,13.563,20.0
4,0.30,226,211,183,102929.0,-10350.0,455.438,-45.796,207,-0.101,91.593,30.0
5,0.32,27,27,27,14491.0,-2390.0,536.704,-88.519,27,-0.165,100.000,32.0
6,0.40,206,185,169,116428.0,-23052.0,565.184,-111.903,180,-0.198,87.379,40.0
7,0.45,11,10,10,5485.0,-2493.0,498.636,-226.636,11,-0.455,100.000,45.0
8,0.50,66,64,64,58912.0,-20508.0,892.606,-310.727,66,-0.348,100.000,50.0
9,0.60,138,127,113,6649.0,-5944.0,48.181,-43.072,138,-0.894,100.000,60.0



2. DISCOUNT AND PROFITABILITY
Correlation: discount vs transaction-line profit: -0.219
Correlation: discount vs transaction-line sales: -0.028
Correlation: discount vs transaction-line profit margin: -0.86

DISCOUNT ECONOMICS ANALYSIS COMPLETE


## does product category change the picture?

In [ ]:
category_discount = (
    df_clean.groupby(
        ["Category", "Discount"],
        observed=True
    )
    .agg(
        Transaction_Lines=("Order ID", "size"),
        Unique_Orders=("Order ID", "nunique"),
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Average_Line_Profit=("Profit", "mean"),
        Loss_Making_Lines=(
            "Profit",
            lambda x: (x < 0).sum()
        )
    )
    .reset_index()
)

# Calculating aggregate margin
category_discount["Profit_Margin"] = np.where(
    category_discount["Total_Sales"] > 0,
    category_discount["Total_Profit"] /
    category_discount["Total_Sales"],
    np.nan
)

# Calculating loss-making percentage
category_discount["Loss_Making_Percentage"] = (
    category_discount["Loss_Making_Lines"] /
    category_discount["Transaction_Lines"] * 100
)

# Converting discount to percentage
category_discount["Discount_Percentage"] = (
    category_discount["Discount"] * 100
)

category_discount = category_discount.sort_values(
    ["Category", "Discount"]
)

print("DISCOUNT ECONOMICS BY CATEGORY")
display(category_discount.round(3))

print("\nCATEGORY SUMMARY")
category_summary = (
    df_clean.groupby("Category")
    .agg(
        Transaction_Lines=("Order ID", "size"),
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum")
    )
    .reset_index()
)

category_summary["Aggregate_Profit_Margin"] = (
    category_summary["Total_Profit"] /
    category_summary["Total_Sales"]
)

display(category_summary.round(3))


DISCOUNT ECONOMICS BY CATEGORY


,Category,Discount,Transaction_Lines,Unique_Orders,Total_Sales,Total_Profit,Average_Line_Profit,Loss_Making_Lines,Profit_Margin,Loss_Making_Percentage,Discount_Percentage
0,Furniture,0.00,836,739,256039.0,58131.0,69.535,0,0.227,0.000,0.0
1,Furniture,0.10,76,71,46640.0,7109.0,93.539,4,0.152,5.263,10.0
2,Furniture,0.15,52,51,27560.0,1416.0,27.231,17,0.051,32.692,15.0
3,Furniture,0.20,615,539,216632.0,6259.0,10.177,165,0.029,26.829,20.0
4,Furniture,0.30,221,207,99172.0,-10676.0,-48.308,206,-0.108,93.213,30.0
5,Furniture,0.32,27,27,14491.0,-2390.0,-88.519,27,-0.165,100.000,32.0
6,Furniture,0.40,75,72,45619.0,-16185.0,-215.800,75,-0.355,100.000,40.0
7,Furniture,0.45,11,10,5485.0,-2493.0,-226.636,11,-0.455,100.000,45.0
8,Furniture,0.50,54,53,20978.0,-12873.0,-238.389,54,-0.614,100.000,50.0
9,Furniture,0.60,138,127,6649.0,-5944.0,-43.072,138,-0.894,100.000,60.0



CATEGORY SUMMARY


,Category,Transaction_Lines,Total_Sales,Total_Profit,Aggregate_Profit_Margin
0,Furniture,2120,741725.0,18456.0,0.025
1,Office Supplies,6026,719127.0,122474.0,0.170
2,Technology,1847,836221.0,145429.0,0.174


# investigating individual product sub-categories

In [ ]:
subcategory_discount = (
    df_clean.groupby(
        ["Sub-Category", "Discount"],
        observed=True
    )
    .agg(
        Transaction_Lines=("Order ID", "size"),
        Unique_Orders=("Order ID", "nunique"),
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Loss_Making_Lines=(
            "Profit",
            lambda x: (x < 0).sum()
        )
    )
    .reset_index()
)

# Aggregating profit margin
subcategory_discount["Profit_Margin"] = np.where(
    subcategory_discount["Total_Sales"] > 0,
    subcategory_discount["Total_Profit"] /
    subcategory_discount["Total_Sales"],
    np.nan
)

# Loss-making percentage
subcategory_discount["Loss_Making_Percentage"] = (
    subcategory_discount["Loss_Making_Lines"] /
    subcategory_discount["Transaction_Lines"] * 100
)

subcategory_discount["Discount_Percentage"] = (
    subcategory_discount["Discount"] * 100
)

subcategory_discount = subcategory_discount.sort_values(
    ["Sub-Category", "Discount"]
)

print("SUB-CATEGORY DISCOUNT ECONOMICS")
display(subcategory_discount.round(3))

# Overall sub-category performance
subcategory_summary = (
    df_clean.groupby("Sub-Category")
    .agg(
        Transaction_Lines=("Order ID", "size"),
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum")
    )
    .reset_index()
)

subcategory_summary["Aggregate_Profit_Margin"] = (
    subcategory_summary["Total_Profit"] /
    subcategory_summary["Total_Sales"]
)

subcategory_summary = subcategory_summary.sort_values(
    "Aggregate_Profit_Margin"
)

print("\nOVERALL PERFORMANCE BY SUB-CATEGORY")
display(subcategory_summary.round(3))


SUB-CATEGORY DISCOUNT ECONOMICS


,Sub-Category,Discount,Transaction_Lines,Unique_Orders,Total_Sales,Total_Profit,Loss_Making_Lines,Profit_Margin,Loss_Making_Percentage,Discount_Percentage
0,Accessories,0.00,471,435,118384.0,35284.0,0,0.298,0.000,0.0
1,Accessories,0.20,304,283,49017.0,6648.0,88,0.136,28.947,20.0
2,Appliances,0.00,271,265,78072.0,23183.0,0,0.297,0.000,0.0
3,Appliances,0.10,16,16,4325.0,1084.0,0,0.251,0.000,10.0
4,Appliances,0.20,112,105,21760.0,2499.0,0,0.115,0.000,20.0
5,Appliances,0.80,67,65,3381.0,-8634.0,67,-2.554,100.000,80.0
6,Art,0.00,498,452,18040.0,5379.0,0,0.298,0.000,0.0
7,Art,0.20,298,279,9097.0,1151.0,0,0.127,0.000,20.0
8,Binders,0.00,337,295,81836.0,39306.0,0,0.480,0.000,0.0
9,Binders,0.20,573,495,85448.0,29419.0,0,0.344,0.000,20.0



OVERALL PERFORMANCE BY SUB-CATEGORY


,Sub-Category,Transaction_Lines,Total_Sales,Total_Profit,Aggregate_Profit_Margin
16,Tables,319,206968.0,-17733.0,-0.086
4,Bookcases,228,114879.0,-3479.0,-0.030
15,Supplies,190,46679.0,-1187.0,-0.025
11,Machines,115,189243.0,3387.0,0.018
5,Chairs,616,328173.0,26598.0,0.081
14,Storage,846,223862.0,21280.0,0.095
13,Phones,889,330047.0,44492.0,0.135
9,Furnishings,957,91705.0,13070.0,0.143
3,Binders,1523,203428.0,30200.0,0.148
1,Appliances,466,107538.0,18132.0,0.169


## statistical testing

In [ ]:
import statsmodels.formula.api as smf

#  Preparing regression dataset
reg_df = df_clean.loc[df_clean["Sales"] > 0].copy()

reg_df["Profit_Margin"] = (
    reg_df["Profit"] / reg_df["Sales"]
)

#  Fit regression model
# 0% discount is the reference group.
# Sub-category controls account for product differences.

model = smf.ols(
    formula=(
        "Profit_Margin ~ "
        "C(Discount, Treatment(reference=0)) + "
        'C(Q("Sub-Category"))'
    ),
    data=reg_df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_df["Order ID"]}
)

#  Extracting discount coefficients only
discount_results = pd.DataFrame({
    "Coefficient": model.params,
    "P_Value": model.pvalues,
    "CI_Lower": model.conf_int()[0],
    "CI_Upper": model.conf_int()[1]
})

discount_results = discount_results[
    discount_results.index.str.contains("C\\(Discount")
].copy()

discount_results["Discount_Level"] = (
    discount_results.index
    .str.extract(r"\[T\.(.*?)\]")[0]
)

discount_results = discount_results[
    [
        "Discount_Level",
        "Coefficient",
        "P_Value",
        "CI_Lower",
        "CI_Upper"
    ]
]

print("REGRESSION RESULTS: DISCOUNT EFFECTS")
display(discount_results.round(4))

#  Model overview
print("\nMODEL OVERVIEW")
print("Observations:", int(model.nobs))
print("Orders (clusters):", reg_df["Order ID"].nunique())
print("R-squared:", round(model.rsquared, 4))
print("Adjusted R-squared:", round(model.rsquared_adj, 4))

print("\nFULL MODEL SUMMARY")
print(model.summary())


REGRESSION RESULTS: DISCOUNT EFFECTS


,Discount_Level,Coefficient,P_Value,CI_Lower,CI_Upper
"C(Discount, Treatment(reference=0))[T.0.1]",NaN,-0.0481,0.0,-0.0666,-0.0296
"C(Discount, Treatment(reference=0))[T.0.15]",NaN,-0.0866,0.0,-0.1240,-0.0491
"C(Discount, Treatment(reference=0))[T.0.2]",NaN,-0.1706,0.0,-0.1753,-0.1659
"C(Discount, Treatment(reference=0))[T.0.3]",NaN,-0.3028,0.0,-0.3177,-0.2879
"C(Discount, Treatment(reference=0))[T.0.32]",NaN,-0.2948,0.0,-0.3430,-0.2466
"C(Discount, Treatment(reference=0))[T.0.4]",NaN,-0.4644,0.0,-0.4845,-0.4444
"C(Discount, Treatment(reference=0))[T.0.45]",NaN,-0.6094,0.0,-0.6600,-0.5588
"C(Discount, Treatment(reference=0))[T.0.5]",NaN,-0.7363,0.0,-0.7753,-0.6973
"C(Discount, Treatment(reference=0))[T.0.6]",NaN,-1.0168,0.0,-1.0635,-0.9700
"C(Discount, Treatment(reference=0))[T.0.7]",NaN,-1.3198,0.0,-1.3383,-1.3013



MODEL OVERVIEW
Observations: 9992
Orders (clusters): 5009
R-squared: 0.9342
Adjusted R-squared: 0.934

FULL MODEL SUMMARY
                            OLS Regression Results                            
Dep. Variable:          Profit_Margin   R-squared:                       0.934
Model:                            OLS   Adj. R-squared:                  0.934
Method:                 Least Squares   F-statistic:                     3623.
Date:                Fri, 30 Oct 2026   Prob (F-statistic):               0.00
Time:                        14:40:15   Log-Likelihood:                 6961.5
No. Observations:                9992   AIC:                        -1.387e+04
Df Residuals:                    9964   BIC:                        -1.367e+04
Df Model:                          27                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|

In [ ]:
# Robustness check
# Inspecting  discount coverage across product sub-categories

discount_coverage = pd.crosstab(
    df_clean["Sub-Category"],
    df_clean["Discount"]
)

print("OBSERVATION COUNTS BY SUB-CATEGORY AND DISCOUNT")
display(discount_coverage)

print("\nDISCOUNT LEVELS WITH FEW OBSERVATIONS")

discount_counts = (
    df_clean.groupby("Discount")
    .size()
    .rename("Transaction_Lines")
    .reset_index()
)

discount_counts["Discount_Percentage"] = (
    discount_counts["Discount"] * 100
)

display(discount_counts)

print("\nSUB-CATEGORY AND DISCOUNT COMBINATIONS WITH FEWER THAN 30 LINES")

sparse_combinations = (
    df_clean.groupby(
        ["Sub-Category", "Discount"],
        observed=True
    )
    .size()
    .reset_index(name="Transaction_Lines")
)

sparse_combinations = sparse_combinations[
    sparse_combinations["Transaction_Lines"] < 30
].sort_values(
    ["Sub-Category", "Discount"]
)

display(sparse_combinations)


OBSERVATION COUNTS BY SUB-CATEGORY AND DISCOUNT


Discount,0.00,0.10,0.15,0.20,0.30,0.32,0.40,0.45,0.50,0.60,0.70,0.80
Sub-Category,,,,,,,,,,,,
Accessories,471,0,0,304,0,0,0,0,0,0,0,0
Appliances,271,16,0,112,0,0,0,0,0,0,0,67
Art,498,0,0,298,0,0,0,0,0,0,0,0
Binders,337,0,0,573,0,0,0,0,0,0,380,233
Bookcases,60,0,52,46,10,27,0,0,18,0,15,0
Chairs,133,76,0,250,157,0,0,0,0,0,0,0
Copiers,22,0,0,37,0,0,9,0,0,0,0,0
Envelopes,152,0,0,102,0,0,0,0,0,0,0,0
Fasteners,128,0,0,89,0,0,0,0,0,0,0,0



DISCOUNT LEVELS WITH FEW OBSERVATIONS


,Discount,Transaction_Lines,Discount_Percentage
0,0.00,4798,0.0
1,0.10,94,10.0
2,0.15,52,15.0
3,0.20,3657,20.0
4,0.30,226,30.0
5,0.32,27,32.0
6,0.40,206,40.0
7,0.45,11,45.0
8,0.50,66,50.0
9,0.60,138,60.0



SUB-CATEGORY AND DISCOUNT COMBINATIONS WITH FEWER THAN 30 LINES


,Sub-Category,Discount,Transaction_Lines
3,Appliances,0.10,16
15,Bookcases,0.30,10
16,Bookcases,0.32,27
17,Bookcases,0.50,18
18,Bookcases,0.70,15
23,Copiers,0.00,22
25,Copiers,0.40,9
35,Machines,0.00,29
36,Machines,0.10,2
38,Machines,0.30,5


In [ ]:
#  Robustness check 2
# Inspect discount variation within individual products

product_discount_counts = (
    df_clean.groupby("Product Name")["Discount"]
    .nunique()
    .rename("Unique_Discount_Levels")
    .reset_index()
)

print("PRODUCT DISCOUNT VARIATION")
display(
    product_discount_counts[
        "Unique_Discount_Levels"
    ].value_counts().sort_index().rename_axis(
        "Number_of_Discount_Levels"
    ).reset_index(name="Number_of_Products")
)

print("\nPRODUCTS WITH ONLY ONE DISCOUNT LEVEL")

single_discount_products = product_discount_counts[
    product_discount_counts["Unique_Discount_Levels"] == 1
]

print(
    "Products with one discount level:",
    len(single_discount_products)
)

print(
    "Products with multiple discount levels:",
    (product_discount_counts["Unique_Discount_Levels"] > 1).sum()
)

print("\nOVERALL PRODUCT COUNT:", df_clean["Product Name"].nunique())


PRODUCT DISCOUNT VARIATION


,Number_of_Discount_Levels,Number_of_Products
0,1,294
1,2,1054
2,3,329
3,4,155
4,5,15
5,6,3



PRODUCTS WITH ONLY ONE DISCOUNT LEVEL
Products with one discount level: 294
Products with multiple discount levels: 1556

OVERALL PRODUCT COUNT: 1850


In [ ]:
 # Product regression diagnostic


check_columns = [
    "Profit_Margin",
    "Sales",
    "Profit",
    "Discount"
]

print("1. NON-FINITE VALUES")
print("=" * 50)

for col in check_columns:
    values = product_reg_df[col].to_numpy(dtype=float)

    print(
        col,
        "| NaN:", np.isnan(values).sum(),
        "| Infinite:", np.isinf(values).sum()
    )

print("\n2. PRODUCT AND YEAR VARIATION")
print("=" * 50)

print("Rows:", len(product_reg_df))
print("Unique products:", product_reg_df["Product Name"].nunique())
print("Unique years:", product_reg_df["Year"].nunique())
print("Unique discounts:", product_reg_df["Discount"].nunique())

print("\n3. MARGIN EXTREMES")
print("=" * 50)

display(
    product_reg_df["Profit_Margin"]
    .describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99])
    .to_frame()
)

print("\n4. PRODUCT DISCOUNT VARIATION")
print("=" * 50)

variation = (
    product_reg_df.groupby("Product Name")["Discount"]
    .nunique()
)

print("Products with one discount:", (variation == 1).sum())
print("Products with multiple discounts:", (variation > 1).sum())

print("\n5. MISSING VALUES")
print("=" * 50)

print(product_reg_df[
    ["Profit_Margin", "Discount", "Product Name", "Year", "Order ID"]
].isna().sum())


1. NON-FINITE VALUES
Profit_Margin | NaN: 0 | Infinite: 0
Sales | NaN: 0 | Infinite: 0
Profit | NaN: 0 | Infinite: 0
Discount | NaN: 0 | Infinite: 0

2. PRODUCT AND YEAR VARIATION
Rows: 9992
Unique products: 1850
Unique years: 4
Unique discounts: 12

3. MARGIN EXTREMES


,Profit_Margin
count,9992.000000
mean,0.120216
std,0.469870
min,-4.000000
1%,-1.769613
5%,-0.779020
50%,0.269528
95%,0.491228
99%,0.500000
max,1.000000



4. PRODUCT DISCOUNT VARIATION
Products with one discount: 294
Products with multiple discounts: 1556

5. MISSING VALUES
Profit_Margin    0
Discount         0
Product Name     0
Year             0
Order ID         0
dtype: int64


In [ ]:
# Extreme margin investigation

extreme_transactions = product_reg_df.copy()

extreme_transactions["Profit_Margin"] = (
    extreme_transactions["Profit"] /
    extreme_transactions["Sales"]
)

print("MOST NEGATIVE PROFIT MARGINS")

display(
    extreme_transactions.nsmallest(
        10, "Profit_Margin"
    )[
        [
            "Order ID",
            "Product Name",
            "Sub-Category",
            "Sales",
            "Profit",
            "Discount",
            "Profit_Margin"
        ]
    ]
)

print("\nHIGHEST PROFIT MARGINS")

display(
    extreme_transactions.nlargest(
        10, "Profit_Margin"
    )[
        [
            "Order ID",
            "Product Name",
            "Sub-Category",
            "Sales",
            "Profit",
            "Discount",
            "Profit_Margin"
        ]
    ]
)


MOST NEGATIVE PROFIT MARGINS


,Order ID,Product Name,Sub-Category,Sales,Profit,Discount,Profit_Margin
2061,CA-2018-165099,Hoover Commercial Lightweight Upright Vacuum,Appliances,1.0,-4.0,0.8,-4.000000
2055,CA-2015-101147,Staple holder,Appliances,2.0,-6.0,0.8,-3.000000
3488,CA-2017-110898,Commercial WindTunnel Clean Air Upright Vacuum...,Appliances,2.0,-6.0,0.8,-3.000000
3516,CA-2015-163748,Hoover Replacement Belts For Soft Guard & Comm...,Appliances,3.0,-9.0,0.8,-3.000000
6618,US-2017-131114,Belkin 6 Outlet Metallic Surge Strip,Appliances,4.0,-12.0,0.8,-3.000000
2691,CA-2018-147207,Hoover Commercial SteamVac,Appliances,5.0,-14.0,0.8,-2.800000
8944,US-2018-124926,Hoover Commercial Soft Guard Upright Vacuum An...,Appliances,9.0,-25.0,0.8,-2.777778
3463,CA-2016-164007,Hoover Shoulder Vac Commercial Portable Vacuum,Appliances,143.0,-394.0,0.8,-2.755245
3489,CA-2017-163398,Acco Smartsocket Color-Coded Six-Outlet AC Ada...,Appliances,26.0,-71.0,0.8,-2.730769
7088,CA-2016-107083,Kensington 6 Outlet SmartSocket Surge Protector,Appliances,25.0,-68.0,0.8,-2.720000



HIGHEST PROFIT MARGINS


,Order ID,Product Name,Sub-Category,Sales,Profit,Discount,Profit_Margin
2360,CA-2018-129378,Wilson Jones Easy Flow II Sheet Lifters,Binders,1.0,1.0,0.2,1.000000
2913,CA-2015-154158,Prang Dustless Chalk Sticks,Art,1.0,1.0,0.2,1.000000
3570,CA-2016-102016,"OIC Binder Clips, Mini, 1/4"" Capacity, Black",Fasteners,1.0,1.0,0.0,1.000000
6970,CA-2017-163153,Prang Dustless Chalk Sticks,Art,1.0,1.0,0.2,1.000000
3572,CA-2016-154900,Avery 518,Labels,3.0,2.0,0.0,0.666667
3574,CA-2018-148691,Avery 520,Labels,3.0,2.0,0.0,0.666667
3576,CA-2015-103331,Avoid Verbal Orders Carbonless Minifold Book,Paper,3.0,2.0,0.0,0.666667
6780,CA-2017-166163,Avery 488,Labels,3.0,2.0,0.0,0.666667
7127,CA-2015-118976,Avery 488,Labels,3.0,2.0,0.0,0.666667
7590,CA-2017-160941,Self-Adhesive Removable Labels,Labels,3.0,2.0,0.0,0.666667


In [ ]:
# Margin sensitivity analysis

sensitivity_df = product_reg_df.copy()

sensitivity_df["Profit_Margin"] = (
    sensitivity_df["Profit"] / sensitivity_df["Sales"]
)

# Flag small-sales transactions
sensitivity_df["Small_Sales"] = (
    sensitivity_df["Sales"] < 5
)

print("SMALL-SALES TRANSACTION AUDIT")
print("=" * 55)

print(
    "Transactions with sales below $5:",
    sensitivity_df["Small_Sales"].sum()
)

print(
    "Percentage of observations:",
    round(
        sensitivity_df["Small_Sales"].mean() * 100,
        2
    ),
    "%"
)

print("\nMARGIN DISTRIBUTION: ALL TRANSACTIONS")

display(
    sensitivity_df["Profit_Margin"].describe(
        percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
    ).to_frame()
)

print("\nMARGIN DISTRIBUTION: SALES >= $5")

display(
    sensitivity_df.loc[
        sensitivity_df["Sales"] >= 5,
        "Profit_Margin"
    ].describe(
        percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
    ).to_frame()
)

print("\nDISCOUNT ECONOMICS: ALL TRANSACTIONS")

all_discount_economics = (
    sensitivity_df.groupby("Discount")
    .agg(
        Lines=("Profit", "size"),
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum")
    )
)

all_discount_economics["Aggregate_Margin"] = (
    all_discount_economics["Profit"] /
    all_discount_economics["Sales"]
)

display(all_discount_economics)

print("\nDISCOUNT ECONOMICS: SALES >= $5")

filtered_discount_economics = (
    sensitivity_df.loc[
        sensitivity_df["Sales"] >= 5
    ]
    .groupby("Discount")
    .agg(
        Lines=("Profit", "size"),
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum")
    )
)

filtered_discount_economics["Aggregate_Margin"] = (
    filtered_discount_economics["Profit"] /
    filtered_discount_economics["Sales"]
)

display(filtered_discount_economics)


SMALL-SALES TRANSACTION AUDIT
Transactions with sales below $5: 424
Percentage of observations: 4.24 %

MARGIN DISTRIBUTION: ALL TRANSACTIONS


,Profit_Margin
count,9992.000000
mean,0.120216
std,0.469870
min,-4.000000
1%,-1.769613
5%,-0.779020
50%,0.269528
95%,0.491228
99%,0.500000
max,1.000000



MARGIN DISTRIBUTION: SALES >= $5


,Profit_Margin
count,9568.000000
mean,0.144138
std,0.426299
min,-2.800000
1%,-1.700000
5%,-0.724138
50%,0.272727
95%,0.489796
99%,0.500000
max,0.600000



DISCOUNT ECONOMICS: ALL TRANSACTIONS


,Lines,Sales,Profit,Aggregate_Margin
Discount,,,,
0.00,4798,1088020.0,320946.0,0.294982
0.10,94,54376.0,9025.0,0.165974
0.15,52,27560.0,1416.0,0.051379
0.20,3657,764636.0,90344.0,0.118153
0.30,226,102929.0,-10350.0,-0.100555
0.32,27,14491.0,-2390.0,-0.164930
0.40,206,116428.0,-23052.0,-0.197994
0.45,11,5485.0,-2493.0,-0.454512
0.50,66,58912.0,-20508.0,-0.348112



DISCOUNT ECONOMICS: SALES >= $5


,Lines,Sales,Profit,Aggregate_Margin
Discount,,,,
0.00,4695,1087678.0,320816.0,0.294955
0.10,94,54376.0,9025.0,0.165974
0.15,52,27560.0,1416.0,0.051379
0.20,3531,764243.0,90263.0,0.118108
0.30,226,102929.0,-10350.0,-0.100555
0.32,27,14491.0,-2390.0,-0.164930
0.40,205,116425.0,-23051.0,-0.197990
0.45,11,5485.0,-2493.0,-0.454512
0.50,66,58912.0,-20508.0,-0.348112


# — PRODUCT FIXED EFFECTS


In [ ]:
# Within-product transformation

within_df = product_reg_df.copy()

# Converting discount to categorical indicator columns.
discount_dummies = pd.get_dummies(
    within_df["Discount"],
    prefix="Discount",
    dtype=float
)

# Removing the 0% discount reference category.
discount_dummies = discount_dummies.drop(
    columns=["Discount_0.0"]
)

# Target variable
y = within_df["Profit_Margin"].astype(float)

# Removing product-level means from the target.
y_within = (
    y - y.groupby(within_df["Product Name"]).transform("mean")
)

# Removing product-level means from each discount indicator.
X_within = discount_dummies.copy()

for col in X_within.columns:
    X_within[col] = (
        X_within[col]
        - X_within[col].groupby(
            within_df["Product Name"]
        ).transform("mean")
    )

# Removing columns with no within-product variation.
X_within = X_within.loc[
    :, X_within.var() > 1e-12
]

# Fitting without an intercept: product means have already been removed.
within_model = sm.OLS(
    y_within,
    X_within
).fit(
    cov_type="cluster",
    cov_kwds={"groups": within_df["Order ID"]}
)

print("PRODUCT FIXED-EFFECTS MODEL")
print("=" * 60)

print("Observations:", int(within_model.nobs))
print("Products:", within_df["Product Name"].nunique())
print("Discount indicators estimated:", X_within.shape[1])
print("Order clusters:", within_df["Order ID"].nunique())

print("\nDISCOUNT COEFFICIENTS")
print("=" * 60)

within_results = pd.DataFrame({
    "Coefficient": within_model.params,
    "P_Value": within_model.pvalues,
    "CI_Lower": within_model.conf_int()[0],
    "CI_Upper": within_model.conf_int()[1]
})

display(within_results)


PRODUCT FIXED-EFFECTS MODEL
Observations: 9992
Products: 1850
Discount indicators estimated: 11
Order clusters: 5009

DISCOUNT COEFFICIENTS


,Coefficient,P_Value,CI_Lower,CI_Upper
Discount_0.1,-0.055292,6.167982e-25,-0.065801,-0.044784
Discount_0.15,-0.091409,6.051811e-13,-0.116295,-0.066524
Discount_0.2,-0.168453,0.000000e+00,-0.171015,-0.165891
Discount_0.3,-0.312544,0.000000e+00,-0.316400,-0.308688
Discount_0.32,-0.301467,7.225523e-53,-0.340076,-0.262857
Discount_0.4,-0.475244,0.000000e+00,-0.482942,-0.467547
Discount_0.45,-0.616766,0.000000e+00,-0.630313,-0.603218
Discount_0.5,-0.753988,0.000000e+00,-0.770949,-0.737027
Discount_0.6,-1.008245,0.000000e+00,-1.029449,-0.987041
Discount_0.7,-1.315602,0.000000e+00,-1.329920,-1.301283


#  — CATEGORY HETEROGENEITY MODEL

In [ ]:
# Does the discount-margin association differ by category?

category_model_df = product_reg_df.copy()

category_model_df["Profit_Margin"] = (
    category_model_df["Profit"] /
    category_model_df["Sales"]
)

category_model = smf.ols(
    formula=(
        'Profit_Margin ~ '
        'C(Discount, Treatment(reference=0)) * '
        'C(Category, Treatment(reference="Furniture")) + '
        'C(Q("Sub-Category"))'
    ),
    data=category_model_df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": category_model_df["Order ID"]}
)

print("CATEGORY HETEROGENEITY REGRESSION")
print("=" * 65)

print("Observations:", int(category_model.nobs))
print("Categories:", category_model_df["Category"].nunique())
print("Order clusters:", category_model_df["Order ID"].nunique())
print("R-squared:", round(category_model.rsquared, 4))

print("\nDISCOUNT AND INTERACTION COEFFICIENTS")
print("=" * 65)

category_results = pd.DataFrame({
    "Coefficient": category_model.params,
    "P_Value": category_model.pvalues,
    "CI_Lower": category_model.conf_int()[0],
    "CI_Upper": category_model.conf_int()[1]
})

category_results = category_results[
    category_results.index.astype(str).str.contains(
        "Discount|Category"
    )
]

display(category_results)


/tmp/ipykernel_15160/1943873936.py:23: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  ).fit(


CATEGORY HETEROGENEITY REGRESSION
Observations: 9992
Categories: 3
Order clusters: 5009
R-squared: 0.9365

DISCOUNT AND INTERACTION COEFFICIENTS


,Coefficient,P_Value,CI_Lower,CI_Upper
"C(Discount, Treatment(reference=0))[T.0.1]",-7.966220e-02,2.083349e-16,-9.866324e-02,-6.066116e-02
"C(Discount, Treatment(reference=0))[T.0.15]",-1.551431e-01,1.837755e-31,-1.812017e-01,-1.290846e-01
"C(Discount, Treatment(reference=0))[T.0.2]",-1.835972e-01,8.636284e-227,-1.947896e-01,-1.724048e-01
"C(Discount, Treatment(reference=0))[T.0.3]",-3.248178e-01,0.000000e+00,-3.406001e-01,-3.090354e-01
"C(Discount, Treatment(reference=0))[T.0.32]",-3.633581e-01,9.612839e-71,-4.034063e-01,-3.233098e-01
"C(Discount, Treatment(reference=0))[T.0.4]",-5.274456e-01,0.000000e+00,-5.523320e-01,-5.025592e-01
"C(Discount, Treatment(reference=0))[T.0.45]",-6.406212e-01,5.158821e-136,-6.912047e-01,-5.900378e-01
"C(Discount, Treatment(reference=0))[T.0.5]",-8.135880e-01,0.000000e+00,-8.420610e-01,-7.851150e-01
"C(Discount, Treatment(reference=0))[T.0.6]",-1.020690e+00,0.000000e+00,-1.067490e+00,-9.738902e-01
"C(Discount, Treatment(reference=0))[T.0.7]",-1.853788e+00,5.469343e-219,-1.968813e+00,-1.738764e+00


# — EXTRACTING CATEGORY-SPECIFIC EFFECTS

In [ ]:
# Corrected design-matrix extraction

import patsy

# Creating  comparison observations
categories = [
    "Furniture",
    "Office Supplies",
    "Technology"
]

prediction_grid = pd.DataFrame({
    "Discount": [0.0, 0.20] * 3,
    "Category": [
        category
        for category in categories
        for _ in range(2)
    ]
})

# Rebuilding the design matrix using the model's formula
formula_rhs = category_compare_model.model.formula.split("~", 1)[1]

X_grid = patsy.dmatrix(
    formula_rhs,
    prediction_grid,
    return_type="dataframe"
)

# Aligning coefficients and covariance matrix with design columns
column_names = X_grid.columns

params = category_compare_model.params.reindex(
    column_names
).to_numpy()

covariance = category_compare_model.cov_params().loc[
    column_names, column_names
].to_numpy()

# Calculating 20% vs 0% margin difference for each category
results = []

for i, category in enumerate(categories):

    contrast = (
        X_grid.iloc[2*i + 1].to_numpy()
        - X_grid.iloc[2*i].to_numpy()
    )

    estimate = contrast @ params

    standard_error = np.sqrt(
        contrast @ covariance @ contrast
    )

    results.append({
        "Category": category,
        "Margin_Difference": estimate,
        "SE": standard_error,
        "CI_Lower": estimate - 1.96 * standard_error,
        "CI_Upper": estimate + 1.96 * standard_error
    })

category_effects = pd.DataFrame(results)

print("CATEGORY-SPECIFIC DISCOUNT EFFECTS")
print("=" * 65)

display(
    category_effects.round(4)
)


CATEGORY-SPECIFIC DISCOUNT EFFECTS


,Category,Margin_Difference,SE,CI_Lower,CI_Upper
0,Furniture,-0.2184,0.0066,-0.2313,-0.2055
1,Office Supplies,-0.1336,0.0046,-0.1427,-0.1245
2,Technology,-0.1777,0.0063,-0.1899,-0.1654


# — Testing whether the category differences are statistically significant

In [ ]:
# Reusing the design matrix and fitted model from the previous cell
# Calculating the contrast vector for each category

effect_vectors = {}

for i, category in enumerate(categories):
    effect_vectors[category] = (
        X_grid.iloc[2*i + 1].to_numpy()
        - X_grid.iloc[2*i].to_numpy()
    )

# Comparing Office Supplies and Technology against Furniture
comparisons = [
    ("Office Supplies vs Furniture", "Office Supplies", "Furniture"),
    ("Technology vs Furniture", "Technology", "Furniture"),
    ("Technology vs Office Supplies", "Technology", "Office Supplies")
]

comparison_results = []

for label, category_a, category_b in comparisons:

    contrast = (
        effect_vectors[category_a]
        - effect_vectors[category_b]
    )

    test = category_compare_model.t_test(contrast)

    comparison_results.append({
        "Comparison": label,
        "Difference_pp": float(np.asarray(test.effect).squeeze()) * 100,
        "P_Value": float(np.asarray(test.pvalue).squeeze())
    })

comparison_table = pd.DataFrame(comparison_results)

print("PAIRWISE CATEGORY DIFFERENCES")
display(comparison_table)

# Overall test: are all three category effects equal?
# Two independent restrictions are sufficient.
R = np.vstack([
    effect_vectors["Office Supplies"] - effect_vectors["Furniture"],
    effect_vectors["Technology"] - effect_vectors["Furniture"]
])

joint_test = category_compare_model.f_test(R)

print("\nOVERALL TEST OF CATEGORY HETEROGENEITY")
print("F-statistic:", float(np.asarray(joint_test.fvalue).squeeze()))
print("P-value:", float(np.asarray(joint_test.pvalue).squeeze()))


PAIRWISE CATEGORY DIFFERENCES


,Comparison,Difference_pp,P_Value
0,Office Supplies vs Furniture,8.475954,5.514623e-25
1,Technology vs Furniture,4.069372,8.074114e-06
2,Technology vs Office Supplies,-4.406581,1.034201e-08



OVERALL TEST OF CATEGORY HETEROGENEITY
F-statistic: 55.959142164869085
P-value: 9.825490407130215e-25


##  Research Finding: Category-Level Heterogeneity in Discount–Profitability Associations

A category-level regression analysis was conducted using transactions with either a 0% or 20% discount. The final sample contained 8,455 transaction lines across 4,535 order clusters.

The estimated association between a 20% discount and profit margin, relative to a 0% discount, differed across the three product categories:

- **Furniture:** −21.84 percentage points (95% CI: −23.13 to −20.55).
- **Office Supplies:** −13.36 percentage points (95% CI: −14.27 to −12.45).
- **Technology:** −17.77 percentage points (95% CI: −18.99 to −16.54).

An overall test of category heterogeneity was statistically significant (F = 55.96, p = 9.83 × 10⁻²⁵). Pairwise comparisons also indicated significant differences between all three category-specific estimates.

These findings suggest that the relationship between discount levels and profitability varies by product category. Furniture displayed the largest negative association, whereas Office Supplies displayed the smallest.

**Interpretation limitation:** The analysis uses observational transaction data. Therefore, the estimates describe associations rather than causal effects. Differences in product mix, pricing decisions and other unobserved factors may influence the results.

##  — RFM CUSTOMER ANALYSIS

In [ ]:
#  Calculating Recency, Frequency and Monetary Value


# Referencing date: one day after the final recorded transaction
snapshot_date = df_orders["Order_Date"].max() + pd.Timedelta(days=1)

# Build customer-level RFM metrics from order-level data
df_rfm = (
    df_orders.groupby("Customer_Name")
    .agg(
        Last_Purchase=("Order_Date", "max"),
        Frequency=("Order ID", "nunique"),
        Monetary=("Sales", "sum"),
        Total_Profit=("Profit", "sum")
    )
    .reset_index()
)

# Calculating days since last purchase
df_rfm["Recency"] = (
    snapshot_date - df_rfm["Last_Purchase"]
).dt.days

# Calculating average order value
df_rfm["Average_Order_Value"] = (
    df_rfm["Monetary"] / df_rfm["Frequency"]
)

# Arranging columns
df_rfm = df_rfm[
    [
        "Customer_Name",
        "Recency",
        "Frequency",
        "Monetary",
        "Total_Profit",
        "Average_Order_Value",
        "Last_Purchase"
    ]
]

# Dataset overview
print("RFM DATASET OVERVIEW")
print("=" * 50)
print("Reference date:", snapshot_date.date())
print("Total customers:", len(df_rfm))
print("Missing values:", df_rfm.isna().sum().sum())
print("Duplicate customers:", df_rfm["Customer_Name"].duplicated().sum())

print("\nRFM DESCRIPTIVE STATISTICS")
display(
    df_rfm[
        ["Recency", "Frequency", "Monetary",
         "Total_Profit", "Average_Order_Value"]
    ].describe().round(2)
)

print("\nFIRST 10 CUSTOMERS")
display(df_rfm.head(10))


RFM DATASET OVERVIEW
Reference date: 2018-12-31
Total customers: 793
Missing values: 0
Duplicate customers: 0

RFM DESCRIPTIVE STATISTICS


,Recency,Frequency,Monetary,Total_Profit,Average_Order_Value
count,793.00,793.00,793.00,793.00,793.00
mean,147.77,6.32,2896.69,361.11,460.13
std,186.11,2.55,2628.51,894.22,433.39
min,1.00,1.00,5.00,-6625.00,2.50
25%,31.00,5.00,1145.00,37.00,213.67
50%,76.00,6.00,2257.00,227.00,362.50
75%,184.00,8.00,3785.00,561.00,550.17
max,1166.00,17.00,25042.00,8983.00,5008.40



FIRST 10 CUSTOMERS


,Customer_Name,Recency,Frequency,Monetary,Total_Profit,Average_Order_Value,Last_Purchase
0,Aaron Bergman,416,3,887.0,129.0,295.666667,2017-11-10
1,Aaron Hawkins,13,7,1744.0,364.0,249.142857,2018-12-18
2,Aaron Smayling,89,7,3051.0,-254.0,435.857143,2018-10-03
3,Adam Bellavance,55,8,7757.0,2055.0,969.625000,2018-11-06
4,Adam Hart,35,10,3251.0,283.0,325.100000,2018-11-26
5,Adam Shillingsburg,29,9,3255.0,63.0,361.666667,2018-12-02
6,Adrian Barton,42,10,14476.0,5445.0,1447.600000,2018-11-19
7,Adrian Hane,61,7,1736.0,-2.0,248.000000,2018-10-31
8,Adrian Shami,42,2,58.0,22.0,29.000000,2018-11-19
9,Aimee Bixby,42,5,969.0,314.0,193.800000,2018-11-19


# — RFM SCORING

In [ ]:
#  Assigning  customer scores from 1 to 5

df_rfm = df_rfm.copy()

# Percentile ranks
recency_rank = df_rfm["Recency"].rank(
    method="average",
    pct=True
)

frequency_rank = df_rfm["Frequency"].rank(
    method="average",
    pct=True
)

monetary_rank = df_rfm["Monetary"].rank(
    method="average",
    pct=True
)

# Convert percentile ranks into scores from 1 to 5
df_rfm["R_Score"] = (
    6 - np.ceil(recency_rank * 5)
).clip(1, 5).astype(int)

df_rfm["F_Score"] = (
    np.ceil(frequency_rank * 5)
).clip(1, 5).astype(int)

df_rfm["M_Score"] = (
    np.ceil(monetary_rank * 5)
).clip(1, 5).astype(int)

# Combined RFM score
df_rfm["RFM_Total"] = (
    df_rfm["R_Score"]
    + df_rfm["F_Score"]
    + df_rfm["M_Score"]
)

# Three-digit RFM score
df_rfm["RFM_Code"] = (
    df_rfm["R_Score"].astype(str)
    + df_rfm["F_Score"].astype(str)
    + df_rfm["M_Score"].astype(str)
)

# Inspecting score distributions
print("RFM SCORE DISTRIBUTIONS")
print("=" * 50)

for column in ["R_Score", "F_Score", "M_Score"]:
    print(f"\n{column}")
    display(
        df_rfm[column]
        .value_counts()
        .sort_index()
        .rename("Customers")
        .to_frame()
    )

print("\nCOMBINED RFM SCORE SUMMARY")
display(
    df_rfm["RFM_Total"].describe().round(2)
)

print("\nCUSTOMER SCORE EXAMPLES")
display(
    df_rfm[
        [
            "Customer_Name",
            "Recency",
            "Frequency",
            "Monetary",
            "R_Score",
            "F_Score",
            "M_Score",
            "RFM_Total",
            "RFM_Code"
        ]
    ].head(15)
)


RFM SCORE DISTRIBUTIONS

R_Score


,Customers
R_Score,
1,159
2,159
3,159
4,154
5,162



F_Score


,Customers
F_Score,
1,195
2,134
3,107
4,198
5,159



M_Score


,Customers
M_Score,
1,158
2,159
3,158
4,158
5,160



COMBINED RFM SCORE SUMMARY


,RFM_Total
count,793.00
mean,8.99
std,3.27
min,3.00
25%,6.00
50%,9.00
75%,11.00
max,15.00



CUSTOMER SCORE EXAMPLES


,Customer_Name,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Total,RFM_Code
0,Aaron Bergman,416,3,887.0,1,1,1,3,111
1,Aaron Hawkins,13,7,1744.0,5,4,3,12,543
2,Aaron Smayling,89,7,3051.0,3,4,4,11,344
3,Adam Bellavance,55,8,7757.0,3,4,5,12,345
4,Adam Hart,35,10,3251.0,4,5,4,13,454
5,Adam Shillingsburg,29,9,3255.0,4,5,4,13,454
6,Adrian Barton,42,10,14476.0,4,5,5,14,455
7,Adrian Hane,61,7,1736.0,3,4,2,9,342
8,Adrian Shami,42,2,58.0,4,1,1,6,411
9,Aimee Bixby,42,5,969.0,4,2,2,8,422


#  — RFM SCORE QUALITY CHECK

In [ ]:
score_columns = ["R_Score", "F_Score", "M_Score"]

print("MISSING SCORE VALUES")
display(df_rfm[score_columns].isna().sum().to_frame("Missing"))

print("\nSCORE RANGE VALIDATION")
for column in score_columns:
    print(
        column,
        "| Minimum:", df_rfm[column].min(),
        "| Maximum:", df_rfm[column].max(),
        "| Valid:", df_rfm[column].between(1, 5).all()
    )

print("\nINCOMPLETE CUSTOMER RECORDS")
display(
    df_rfm[df_rfm[score_columns].isna().any(axis=1)]
)

# Verifying combined scores
expected_total = df_rfm[score_columns].sum(axis=1)

print(
    "\nCombined score consistency:",
    df_rfm["RFM_Total"].equals(expected_total)
)

print(
    "Total customers:",
    len(df_rfm)
)


MISSING SCORE VALUES


,Missing
R_Score,0
F_Score,0
M_Score,0



SCORE RANGE VALIDATION
R_Score | Minimum: 1 | Maximum: 5 | Valid: True
F_Score | Minimum: 1 | Maximum: 5 | Valid: True
M_Score | Minimum: 1 | Maximum: 5 | Valid: True

INCOMPLETE CUSTOMER RECORDS


,Customer_Name,Recency,Frequency,Monetary,Total_Profit,Average_Order_Value,Last_Purchase,R_Score,F_Score,M_Score,RFM_Total,RFM_Code



Combined score consistency: True
Total customers: 793


##  — CUSTOMER SEGMENTATION

In [ ]:
def assign_segment(row):

    r = row["R_Score"]
    f = row["F_Score"]
    m = row["M_Score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"

    elif r >= 3 and f >= 4:
        return "Loyal Customers"

    elif r >= 4 and f in [2, 3]:
        return "Potential Loyalists"

    elif r >= 4 and f == 1:
        return "New Customers"

    elif r <= 2 and f >= 4:
        return "At Risk"

    elif r <= 2 and f in [2, 3]:
        return "Needs Attention"

    elif r <= 2 and f == 1:
        return "Hibernating"

    else:
        return "Regular Customers"


df_rfm["Customer_Segment"] = df_rfm.apply(
    assign_segment,
    axis=1
)

# Customer counts and proportions
segment_summary = (
    df_rfm.groupby("Customer_Segment")
    .agg(
        Customers=("Customer_Name", "count"),
        Average_Recency=("Recency", "mean"),
        Average_Frequency=("Frequency", "mean"),
        Average_Monetary=("Monetary", "mean"),
        Average_Profit=("Total_Profit", "mean")
    )
    .reset_index()
)

segment_summary["Customer_Percentage"] = (
    segment_summary["Customers"] /
    len(df_rfm) * 100
)

segment_summary = segment_summary.sort_values(
    "Customers",
    ascending=False
)

print("CUSTOMER SEGMENT DISTRIBUTION")
display(segment_summary.round(2))

print("\nSEGMENT VALIDATION")
print("Total customers:", df_rfm["Customer_Name"].nunique())
print("Customers assigned:", df_rfm["Customer_Segment"].notna().sum())
print("Unassigned customers:", df_rfm["Customer_Segment"].isna().sum())
print("Number of segments:", df_rfm["Customer_Segment"].nunique())


CUSTOMER SEGMENT DISTRIBUTION


,Customer_Segment,Customers,Average_Recency,Average_Frequency,Average_Monetary,Average_Profit,Customer_Percentage
3,Loyal Customers,157,53.22,8.24,2756.03,284.15,19.80
4,Needs Attention,115,264.07,5.41,2318.27,327.51,14.50
2,Hibernating,113,428.58,3.05,1275.15,70.31,14.25
1,Champions,110,25.45,9.22,5226.96,615.42,13.87
0,At Risk,90,199.97,8.52,4169.99,727.71,11.35
6,Potential Loyalists,85,26.92,5.49,2806.89,409.28,10.72
7,Regular Customers,76,74.72,4.49,2585.97,192.11,9.58
5,New Customers,47,26.96,3.40,1453.15,288.51,5.93



SEGMENT VALIDATION
Total customers: 793
Customers assigned: 793
Unassigned customers: 0
Number of segments: 8


 # — FINANCIAL VALUE OF CUSTOMER SEGMENTS


In [ ]:
segment_financials = (
    df_rfm.groupby("Customer_Segment")
    .agg(
        Customers=("Customer_Name", "count"),
        Total_Sales=("Monetary", "sum"),
        Total_Profit=("Total_Profit", "sum"),
        Average_Customer_Profit=("Total_Profit", "mean"),
        Average_Customer_Sales=("Monetary", "mean"),
        Loss_Making_Customers=(
            "Total_Profit",
            lambda x: (x < 0).sum()
        )
    )
    .reset_index()
)

# Aggregating profit margin
segment_financials["Profit_Margin"] = np.where(
    segment_financials["Total_Sales"] > 0,
    segment_financials["Total_Profit"] /
    segment_financials["Total_Sales"],
    np.nan
)

# Share of total business sales and profit
overall_sales = df_rfm["Monetary"].sum()
overall_profit = df_rfm["Total_Profit"].sum()

segment_financials["Sales_Share_Percent"] = (
    segment_financials["Total_Sales"] /
    overall_sales * 100
)

segment_financials["Profit_Share_Percent"] = (
    segment_financials["Total_Profit"] /
    overall_profit * 100
)

segment_financials["Loss_Making_Percent"] = (
    segment_financials["Loss_Making_Customers"] /
    segment_financials["Customers"] * 100
)

# Sorting by total profit
segment_financials = segment_financials.sort_values(
    "Total_Profit",
    ascending=False
)

print("FINANCIAL PERFORMANCE BY CUSTOMER SEGMENT")
print("=" * 65)

display(
    segment_financials.round(3)
)

print("\nRECONCILIATION")
print("Total segment sales:", round(segment_financials["Total_Sales"].sum(), 2))
print("Total segment profit:", round(segment_financials["Total_Profit"].sum(), 2))
print("Overall customer sales:", round(overall_sales, 2))
print("Overall customer profit:", round(overall_profit, 2))


FINANCIAL PERFORMANCE BY CUSTOMER SEGMENT


,Customer_Segment,Customers,Total_Sales,Total_Profit,Average_Customer_Profit,Average_Customer_Sales,Loss_Making_Customers,Profit_Margin,Sales_Share_Percent,Profit_Share_Percent,Loss_Making_Percent
1,Champions,110,574966.0,67696.0,615.418,5226.964,16,0.118,25.030,23.640,14.545
0,At Risk,90,375299.0,65494.0,727.711,4169.989,11,0.175,16.338,22.871,12.222
3,Loyal Customers,157,432697.0,44611.0,284.146,2756.032,28,0.103,18.837,15.579,17.834
4,Needs Attention,115,266601.0,37664.0,327.513,2318.270,25,0.141,11.606,13.153,21.739
6,Potential Loyalists,85,238586.0,34789.0,409.282,2806.894,17,0.146,10.387,12.149,20.000
7,Regular Customers,76,196534.0,14600.0,192.105,2585.974,16,0.074,8.556,5.098,21.053
5,New Customers,47,68298.0,13560.0,288.511,1453.149,8,0.199,2.973,4.735,17.021
2,Hibernating,113,144092.0,7945.0,70.310,1275.150,34,0.055,6.273,2.774,30.088



RECONCILIATION
Total segment sales: 2297073.0
Total segment profit: 286359.0
Overall customer sales: 2297073.0
Overall customer profit: 286359.0


#  — AT-RISK CUSTOMER PRIORITISATION

In [ ]:
# Extracting At Risk customers
at_risk_df = df_rfm[
    df_rfm["Customer_Segment"] == "At Risk"
].copy()

# Calculating historical customer profit margin
at_risk_df["Customer_Profit_Margin"] = np.where(
    at_risk_df["Monetary"] > 0,
    at_risk_df["Total_Profit"] / at_risk_df["Monetary"],
    np.nan
)

# Classifying historical profitability
at_risk_df["Profitability_Status"] = np.select(
    [
        at_risk_df["Total_Profit"] > 0,
        at_risk_df["Total_Profit"] < 0
    ],
    [
        "Profitable",
        "Loss-making"
    ],
    default="Break-even"
)

# Ranking customers by historical profit
at_risk_df = at_risk_df.sort_values(
    "Total_Profit",
    ascending=False
)

# Displaying highest historical profit contributors
print("TOP 15 AT-RISK CUSTOMERS BY HISTORICAL PROFIT")
display(
    at_risk_df[
        [
            "Customer_Name",
            "Recency",
            "Frequency",
            "Monetary",
            "Total_Profit",
            "Customer_Profit_Margin",
            "Profitability_Status"
        ]
    ].head(15).round(3)
)

# Segment-level diagnostic
print("\nAT-RISK PROFITABILITY DISTRIBUTION")
display(
    at_risk_df["Profitability_Status"]
    .value_counts()
    .to_frame("Customers")
)

print("\nAT-RISK SUMMARY")
display(
    at_risk_df[
        [
            "Recency",
            "Frequency",
            "Monetary",
            "Total_Profit",
            "Customer_Profit_Margin"
        ]
    ].describe().round(2)
)


TOP 15 AT-RISK CUSTOMERS BY HISTORICAL PROFIT


,Customer_Name,Recency,Frequency,Monetary,Total_Profit,Customer_Profit_Margin,Profitability_Status
671,Sanjit Chand,350,9,14145.0,5756.0,0.407,Profitable
431,Keith Dawkins,113,12,8184.0,3042.0,0.372,Profitable
758,Tom Boeckenhauer,203,7,9134.0,2798.0,0.306,Profitable
561,Nathan Mautz,161,7,6461.0,2752.0,0.426,Profitable
327,Helen Wasserman,118,8,9299.0,2163.0,0.233,Profitable
450,Laura Armstrong,155,11,8674.0,2059.0,0.237,Profitable
714,Steven Roelle,197,8,4345.0,1988.0,0.458,Profitable
697,Shirley Daniels,201,9,6121.0,1985.0,0.324,Profitable
169,Clay Ludtke,285,12,10881.0,1937.0,0.178,Profitable
601,Penelope Sewall,237,7,6842.0,1741.0,0.254,Profitable



AT-RISK PROFITABILITY DISTRIBUTION


,Customers
Profitability_Status,
Profitable,79
Loss-making,11



AT-RISK SUMMARY


,Recency,Frequency,Monetary,Total_Profit,Customer_Profit_Margin
count,90.00,90.00,90.00,90.00,90.00
mean,199.97,8.52,4169.99,727.71,0.15
std,105.90,1.57,2750.45,954.14,0.14
min,104.00,7.00,432.00,-1852.00,-0.34
25%,123.00,7.00,2058.00,174.75,0.08
50%,163.00,8.00,3398.50,576.00,0.18
75%,218.00,9.00,5454.00,1113.25,0.24
max,719.00,13.00,14145.00,5756.00,0.46


#  — PERSONALIZED INACTIVITY ANALYSIS

In [ ]:
# Sorting all customer orders chronologically
ordered_history = df_orders.sort_values(
    ["Customer_Name", "Order_Date"]
).copy()

# Calculating days between consecutive orders
ordered_history["Days_Between_Orders"] = (
    ordered_history.groupby("Customer_Name")["Order_Date"]
    .diff()
    .dt.days
)

# Calculating each customer's typical purchasing interval
purchase_cadence = (
    ordered_history.groupby("Customer_Name")
    .agg(
        Median_Order_Interval=("Days_Between_Orders", "median"),
        Mean_Order_Interval=("Days_Between_Orders", "mean"),
        Observed_Intervals=("Days_Between_Orders", "count")
    )
    .reset_index()
)

# Attach historical cadence to At Risk customers
at_risk_cadence = at_risk_df.merge(
    purchase_cadence,
    on="Customer_Name",
    how="left"
)

# Calculating inactivity relative to normal purchasing behaviour
at_risk_cadence["Days_Beyond_Typical_Interval"] = (
    at_risk_cadence["Recency"]
    - at_risk_cadence["Median_Order_Interval"]
)

at_risk_cadence["Inactivity_Ratio"] = (
    at_risk_cadence["Recency"]
    / at_risk_cadence["Median_Order_Interval"]
)

# Classifying relative inactivity
at_risk_cadence["Cadence_Status"] = np.select(
    [
        at_risk_cadence["Inactivity_Ratio"] >= 2,
        at_risk_cadence["Inactivity_Ratio"] >= 1
    ],
    [
        "At least 2x typical interval",
        "Beyond typical interval"
    ],
    default="Within typical interval"
)

print("AT-RISK CUSTOMER PURCHASING CADENCE")
print("=" * 60)

display(
    at_risk_cadence[
        [
            "Customer_Name",
            "Recency",
            "Median_Order_Interval",
            "Days_Beyond_Typical_Interval",
            "Inactivity_Ratio",
            "Total_Profit",
            "Cadence_Status"
        ]
    ]
    .sort_values(
        ["Inactivity_Ratio", "Total_Profit"],
        ascending=[False, False]
    )
    .head(20)
    .round(2)
)

print("\nCADENCE STATUS DISTRIBUTION")

display(
    at_risk_cadence["Cadence_Status"]
    .value_counts()
    .to_frame("Customers")
)

print("\nCADENCE SUMMARY")

display(
    at_risk_cadence[
        [
            "Recency",
            "Median_Order_Interval",
            "Days_Beyond_Typical_Interval",
            "Inactivity_Ratio"
        ]
    ].describe().round(2)
)


AT-RISK CUSTOMER PURCHASING CADENCE


,Customer_Name,Recency,Median_Order_Interval,Days_Beyond_Typical_Interval,Inactivity_Ratio,Total_Profit,Cadence_Status
54,Logan Haushalter,386,42.0,344.0,9.19,316.0,At least 2x typical interval
43,Mark Packer,719,86.0,633.0,8.36,600.0,At least 2x typical interval
24,Jonathan Doherty,395,47.5,347.5,8.32,1053.0,At least 2x typical interval
19,Jim Kriz,181,25.5,155.5,7.10,1172.0,At least 2x typical interval
61,Kean Takahito,392,55.5,336.5,7.06,253.0,At least 2x typical interval
15,Anna Häberlin,411,59.0,352.0,6.97,1298.0,At least 2x typical interval
31,Maribeth Schnelling,274,46.0,228.0,5.96,845.0,At least 2x typical interval
60,Tim Brockman,350,59.5,290.5,5.88,260.0,At least 2x typical interval
79,Odella Nelson,411,70.0,341.0,5.87,-6.0,At least 2x typical interval
16,Pierre Wener,465,80.0,385.0,5.81,1292.0,At least 2x typical interval



CADENCE STATUS DISTRIBUTION


,Customers
Cadence_Status,
At least 2x typical interval,38
Beyond typical interval,38
Within typical interval,14



CADENCE SUMMARY


,Recency,Median_Order_Interval,Days_Beyond_Typical_Interval,Inactivity_Ratio
count,90.00,90.00,90.00,90.00
mean,199.97,103.42,96.55,2.43
std,105.90,42.71,122.10,1.95
min,104.00,25.00,-100.50,0.52
25%,123.00,70.75,25.62,1.20
50%,163.00,95.50,61.50,1.67
75%,218.00,126.88,128.25,2.82
max,719.00,216.00,633.00,9.19


# — RETENTION PRIORITY FRAMEWORK

In [ ]:
at_risk_cadence["Retention_Priority"] = np.select(
    [
        (
            (at_risk_cadence["Total_Profit"] > 0) &
            (at_risk_cadence["Inactivity_Ratio"] >= 2)
        ),
        (
            (at_risk_cadence["Total_Profit"] > 0) &
            (at_risk_cadence["Inactivity_Ratio"] >= 1) &
            (at_risk_cadence["Inactivity_Ratio"] < 2)
        ),
        (
            at_risk_cadence["Total_Profit"] <= 0
        ),
        (
            (at_risk_cadence["Total_Profit"] > 0) &
            (at_risk_cadence["Inactivity_Ratio"] < 1)
        )
    ],
    [
        "High-priority re-engagement",
        "Re-engagement nurture",
        "Profitability review",
        "Within typical cadence"
    ],
    default="Unclassified"
)

# Summarising each priority group
priority_summary = (
    at_risk_cadence.groupby("Retention_Priority")
    .agg(
        Customers=("Customer_Name", "count"),
        Total_Historical_Sales=("Monetary", "sum"),
        Total_Historical_Profit=("Total_Profit", "sum"),
        Average_Recency=("Recency", "mean"),
        Average_Inactivity_Ratio=("Inactivity_Ratio", "mean"),
        Average_Frequency=("Frequency", "mean")
    )
    .reset_index()
)

priority_summary["Customer_Share_Percent"] = (
    priority_summary["Customers"] /
    len(at_risk_cadence) * 100
)

print("AT-RISK RETENTION PRIORITY SUMMARY")
display(priority_summary.round(2))

print("\nVALIDATION")
print("Total At Risk customers:", len(at_risk_cadence))
print("Assigned customers:", at_risk_cadence["Retention_Priority"].notna().sum())
print("Unclassified:", (at_risk_cadence["Retention_Priority"] == "Unclassified").sum())

print("\nHIGH-PRIORITY CUSTOMER LIST")

display(
    at_risk_cadence[
        at_risk_cadence["Retention_Priority"] ==
        "High-priority re-engagement"
    ][
        [
            "Customer_Name",
            "Recency",
            "Median_Order_Interval",
            "Inactivity_Ratio",
            "Frequency",
            "Monetary",
            "Total_Profit"
        ]
    ].sort_values(
        "Total_Profit",
        ascending=False
    ).head(20).round(2)
)


AT-RISK RETENTION PRIORITY SUMMARY


,Retention_Priority,Customers,Total_Historical_Sales,Total_Historical_Profit,Average_Recency,Average_Inactivity_Ratio,Average_Frequency,Customer_Share_Percent
0,High-priority re-engagement,33,164332.0,33384.0,277.82,4.13,9.00,36.67
1,Profitability review,11,31946.0,-4468.0,178.36,2.34,9.27,12.22
2,Re-engagement nurture,33,130248.0,25662.0,155.06,1.41,8.15,36.67
3,Within typical cadence,13,48773.0,10916.0,134.62,0.81,7.62,14.44



VALIDATION
Total At Risk customers: 90
Assigned customers: 90
Unclassified: 0

HIGH-PRIORITY CUSTOMER LIST


,Customer_Name,Recency,Median_Order_Interval,Inactivity_Ratio,Frequency,Monetary,Total_Profit
0,Sanjit Chand,350,122.0,2.87,9,14145.0,5756.0
1,Keith Dawkins,113,25.0,4.52,12,8184.0,3042.0
8,Clay Ludtke,285,52.0,5.48,12,10881.0,1937.0
9,Penelope Sewall,237,74.0,3.20,7,6842.0,1741.0
10,Rick Wilson,121,57.0,2.12,7,7400.0,1586.0
12,Dennis Pardue,359,83.0,4.33,9,5482.0,1571.0
14,Edward Hooks,136,54.0,2.52,12,10313.0,1392.0
15,Anna Häberlin,411,59.0,6.97,12,7889.0,1298.0
16,Pierre Wener,465,80.0,5.81,7,3922.0,1292.0
19,Jim Kriz,181,25.5,7.10,9,4762.0,1172.0


# — CUSTOMER VALUE RELATIONSHIPS

In [ ]:
from scipy.stats import spearmanr, pearsonr

analysis_df = df_rfm.copy()

# Calculating customer-level profit margin
analysis_df["Customer_Profit_Margin"] = (
    analysis_df["Total_Profit"] /
    analysis_df["Monetary"]
)

# Variables to investigate
relationships = [
    ("Frequency", "Total_Profit"),
    ("Monetary", "Total_Profit"),
    ("Frequency", "Monetary"),
    ("Monetary", "Customer_Profit_Margin"),
    ("Frequency", "Customer_Profit_Margin")
]

results = []

for x, y in relationships:

    spearman_r, spearman_p = spearmanr(
        analysis_df[x],
        analysis_df[y]
    )

    pearson_r, pearson_p = pearsonr(
        analysis_df[x],
        analysis_df[y]
    )

    results.append({
        "Relationship": f"{x} vs {y}",
        "Spearman_Correlation": spearman_r,
        "Spearman_P_Value": spearman_p,
        "Pearson_Correlation": pearson_r,
        "Pearson_P_Value": pearson_p
    })

correlation_results = pd.DataFrame(results)

print("CUSTOMER VALUE CORRELATION ANALYSIS")
print("=" * 65)

display(correlation_results.round(4))

# Comparing frequency groups
analysis_df["Frequency_Group"] = pd.cut(
    analysis_df["Frequency"],
    bins=[0, 3, 6, 9, np.inf],
    labels=["1–3 orders", "4–6 orders", "7–9 orders", "10+ orders"]
)

frequency_profit_summary = (
    analysis_df.groupby("Frequency_Group", observed=False)
    .agg(
        Customers=("Customer_Name", "count"),
        Average_Sales=("Monetary", "mean"),
        Average_Profit=("Total_Profit", "mean"),
        Median_Profit=("Total_Profit", "median"),
        Total_Profit=("Total_Profit", "sum"),
        Average_Profit_Margin=("Customer_Profit_Margin", "mean")
    )
    .reset_index()
)

print("\nPURCHASING FREQUENCY AND PROFITABILITY")
display(frequency_profit_summary.round(3))


CUSTOMER VALUE CORRELATION ANALYSIS


,Relationship,Spearman_Correlation,Spearman_P_Value,Pearson_Correlation,Pearson_P_Value
0,Frequency vs Total_Profit,0.3246,0.0000,0.1747,0.0000
1,Monetary vs Total_Profit,0.5392,0.0000,0.5064,0.0000
2,Frequency vs Monetary,0.5621,0.0000,0.4177,0.0000
3,Monetary vs Customer_Profit_Margin,0.0383,0.2810,0.0831,0.0193
4,Frequency vs Customer_Profit_Margin,0.0105,0.7681,0.0434,0.2217



PURCHASING FREQUENCY AND PROFITABILITY


,Frequency_Group,Customers,Average_Sales,Average_Profit,Median_Profit,Total_Profit,Average_Profit_Margin
0,1–3 orders,99,1051.475,54.434,56.0,5389.0,0.092
1,4–6 orders,337,2403.605,306.139,186.0,103169.0,0.104
2,7–9 orders,269,3489.762,440.810,338.0,118578.0,0.122
3,10+ orders,88,5047.909,672.989,527.0,59223.0,0.119


#
#— TEMPORAL MODELLING PREPARATION


In [ ]:
print("DATASET TIME COVERAGE")
print("=" * 55)

print("First order:", df_orders["Order_Date"].min())
print("Last order:", df_orders["Order_Date"].max())

monthly_orders = (
    df_orders.assign(
        Order_Month=df_orders["Order_Date"].dt.to_period("M")
    )
    .groupby("Order_Month")
    .agg(
        Orders=("Order ID", "nunique"),
        Customers=("Customer_Name", "nunique"),
        Sales=("Sales", "sum")
    )
    .reset_index()
)

print("\nMONTHLY ORDER COVERAGE")
display(monthly_orders)

print("\nTOTAL MONTHS:", len(monthly_orders))
print("TOTAL ORDERS:", df_orders["Order ID"].nunique())
print("TOTAL CUSTOMERS:", df_orders["Customer_Name"].nunique())


DATASET TIME COVERAGE
First order: 2015-01-03 00:00:00
Last order: 2018-12-30 00:00:00

MONTHLY ORDER COVERAGE


,Order_Month,Orders,Customers,Sales
0,2015-01,32,32,14238.0
1,2015-02,28,27,4518.0
2,2015-03,71,69,55689.0
3,2015-04,66,64,28013.0
4,2015-05,69,67,23648.0
5,2015-06,66,63,34598.0
6,2015-07,65,65,33948.0
7,2015-08,72,70,27908.0
8,2015-09,130,118,81787.0
9,2015-10,78,75,31448.0



TOTAL MONTHS: 48
TOTAL ORDERS: 5009
TOTAL CUSTOMERS: 793


# — TEMPORAL COHORT FEASIBILITY CHECK

In [ ]:
snapshot_dates = [
    "2016-03-31",
    "2016-06-30",
    "2016-09-30",
    "2016-12-31",
    "2017-03-31",
    "2017-06-30",
    "2017-09-30",
    "2017-12-31",
    "2018-03-31",
    "2018-06-30",
    "2018-09-30"
]

snapshot_dates = pd.to_datetime(snapshot_dates)

cohort_results = []

for snapshot in snapshot_dates:

    # Customer activity during the previous 365 days
    history_start = snapshot - pd.Timedelta(days=365)

    historical_orders = df_orders[
        (df_orders["Order_Date"] > history_start) &
        (df_orders["Order_Date"] <= snapshot)
    ]

    eligible_customers = set(
        historical_orders["Customer_Name"].unique()
    )

    # Customer activity during the following 90 days
    future_end = snapshot + pd.Timedelta(days=90)

    future_orders = df_orders[
        (df_orders["Order_Date"] > snapshot) &
        (df_orders["Order_Date"] <= future_end)
    ]

    future_customers = set(
        future_orders["Customer_Name"].unique()
    )

    # Repeat-purchase labels
    repeat_customers = eligible_customers.intersection(
        future_customers
    )

    total_eligible = len(eligible_customers)
    repeat_count = len(repeat_customers)

    cohort_results.append({
        "Snapshot_Date": snapshot.date(),
        "Eligible_Customers": total_eligible,
        "Repeat_Purchasers": repeat_count,
        "Non_Repeat_Purchasers": total_eligible - repeat_count,
        "Repeat_Rate_Percent": (
            repeat_count / total_eligible * 100
            if total_eligible > 0 else np.nan
        )
    })

cohort_summary = pd.DataFrame(cohort_results)

print("TEMPORAL COHORT FEASIBILITY")
print("=" * 65)

display(cohort_summary.round(2))

print("\nTOTAL SNAPSHOTS:", len(cohort_summary))
print("TOTAL CUSTOMER-SNAPSHOT OBSERVATIONS:",
      cohort_summary["Eligible_Customers"].sum())

print("\nTRAINING PERIOD")
display(cohort_summary.iloc[:5].round(2))

print("\nVALIDATION PERIOD")
display(cohort_summary.iloc[5:8].round(2))

print("\nTEST PERIOD")
display(cohort_summary.iloc[8:].round(2))


TEMPORAL COHORT FEASIBILITY


,Snapshot_Date,Eligible_Customers,Repeat_Purchasers,Non_Repeat_Purchasers,Repeat_Rate_Percent
0,2016-03-31,579,140,439,24.18
1,2016-06-30,576,176,400,30.56
2,2016-09-30,568,226,342,39.79
3,2016-12-31,573,115,458,20.07
4,2017-03-31,589,166,423,28.18
5,2017-06-30,622,230,392,36.98
6,2017-09-30,631,266,365,42.16
7,2017-12-31,638,177,461,27.74
8,2018-03-31,645,240,405,37.21
9,2018-06-30,656,289,367,44.05



TOTAL SNAPSHOTS: 11
TOTAL CUSTOMER-SNAPSHOT OBSERVATIONS: 6743

TRAINING PERIOD


,Snapshot_Date,Eligible_Customers,Repeat_Purchasers,Non_Repeat_Purchasers,Repeat_Rate_Percent
0,2016-03-31,579,140,439,24.18
1,2016-06-30,576,176,400,30.56
2,2016-09-30,568,226,342,39.79
3,2016-12-31,573,115,458,20.07
4,2017-03-31,589,166,423,28.18



VALIDATION PERIOD


,Snapshot_Date,Eligible_Customers,Repeat_Purchasers,Non_Repeat_Purchasers,Repeat_Rate_Percent
5,2017-06-30,622,230,392,36.98
6,2017-09-30,631,266,365,42.16
7,2017-12-31,638,177,461,27.74



TEST PERIOD


,Snapshot_Date,Eligible_Customers,Repeat_Purchasers,Non_Repeat_Purchasers,Repeat_Rate_Percent
8,2018-03-31,645,240,405,37.21
9,2018-06-30,656,289,367,44.05
10,2018-09-30,666,369,297,55.41


# — CUSTOMER PREDICTION DATASET

In [ ]:
model_records = []

for snapshot in snapshot_dates:

    history_start = snapshot - pd.Timedelta(days=365)
    future_end = snapshot + pd.Timedelta(days=90)

    #     Historical order data: strictly before or on snapshot
    history = df_orders[
        (df_orders["Order_Date"] > history_start) &
        (df_orders["Order_Date"] <= snapshot)
    ].copy()

    #    Future orders: used ONLY to create the target
    future = df_orders[
        (df_orders["Order_Date"] > snapshot) &
        (df_orders["Order_Date"] <= future_end)
    ].copy()

    #    Historical customer-level features
    customer_features = (
        history.groupby("Customer_Name")
        .agg(
            Order_Frequency=("Order ID", "nunique"),
            Total_Sales=("Sales", "sum"),
            Total_Profit=("Profit", "sum"),
            Average_Order_Value=("Sales", "mean"),
            Average_Order_Profit=("Profit", "mean"),
            Last_Order_Date=("Order_Date", "max"),
            Segment=("Segment", lambda x: x.mode().iloc[0])
        )
        .reset_index()
    )

    customer_features["Recency_Days"] = (
        snapshot - customer_features["Last_Order_Date"]
    ).dt.days

    #    Historical purchasing intervals
    intervals = (
        history.sort_values(["Customer_Name", "Order_Date"])
        .groupby("Customer_Name")["Order_Date"]
        .apply(lambda x: x.diff().dt.days.median())
        .rename("Median_Order_Interval")
        .reset_index()
    )

    customer_features = customer_features.merge(
        intervals, on="Customer_Name", how="left"
    )

    #    Historical line-level features
    historical_lines = df_clean[
        (df_clean["Order Date"] > history_start) &
        (df_clean["Order Date"] <= snapshot)
    ].copy()

    line_features = (
        historical_lines.groupby("Customer Name")
        .agg(
            Average_Discount=("Discount", "mean"),
            Maximum_Discount=("Discount", "max"),
            Unique_Products=("Product Name", "nunique"),
            Unique_Categories=("Category", "nunique"),
            Unique_Subcategories=("Sub-Category", "nunique")
        )
        .reset_index()
        .rename(columns={"Customer Name": "Customer_Name"})
    )

    customer_features = customer_features.merge(
        line_features, on="Customer_Name", how="left"
    )

    #     Target: at least one purchase in the next 90 days
    future_customers = set(future["Customer_Name"].unique())

    customer_features["Repeat_Within_90_Days"] = (
        customer_features["Customer_Name"]
        .isin(future_customers)
        .astype(int)
    )

    #    Storing the snapshot date
    customer_features["Snapshot_Date"] = snapshot

    model_records.append(customer_features)

#       Combine all snapshots
signal_model_df = pd.concat(
    model_records, ignore_index=True
)

# Removing the raw last-order timestamp:
# Recency_Days already captures the useful information.
signal_model_df = signal_model_df.drop(
    columns=["Last_Order_Date"]
)

print("SIGNAL MODELLING DATASET")
print("=" * 60)

print("Shape:", signal_model_df.shape)
print("Unique customers:",
      signal_model_df["Customer_Name"].nunique())
print("Unique snapshots:",
      signal_model_df["Snapshot_Date"].nunique())

print("\nTARGET DISTRIBUTION")
display(
    signal_model_df["Repeat_Within_90_Days"]
    .value_counts()
    .rename(index={0: "No repeat", 1: "Repeat"})
    .to_frame("Observations")
)

print("\nTARGET RATE BY SNAPSHOT")
display(
    signal_model_df.groupby("Snapshot_Date")
    ["Repeat_Within_90_Days"]
    .mean()
    .mul(100)
    .round(2)
    .rename("Repeat_Rate_Percent")
    .reset_index()
)

print("\nMISSING VALUES")
display(
    signal_model_df.isna().sum()
    .to_frame("Missing_Values")
    .query("Missing_Values > 0")
)

print("\nDUPLICATE CUSTOMER-SNAPSHOT RECORDS:",
      signal_model_df.duplicated(
          ["Customer_Name", "Snapshot_Date"]
      ).sum())

print("\nFEATURE PREVIEW")
display(signal_model_df.head())


SIGNAL MODELLING DATASET
Shape: (6743, 16)
Unique customers: 789
Unique snapshots: 11

TARGET DISTRIBUTION


,Observations
Repeat_Within_90_Days,
No repeat,4349
Repeat,2394



TARGET RATE BY SNAPSHOT


,Snapshot_Date,Repeat_Rate_Percent
0,2016-03-31,24.18
1,2016-06-30,30.56
2,2016-09-30,39.79
3,2016-12-31,20.07
4,2017-03-31,28.18
5,2017-06-30,36.98
6,2017-09-30,42.16
7,2017-12-31,27.74
8,2018-03-31,37.21
9,2018-06-30,44.05



MISSING VALUES


,Missing_Values
Median_Order_Interval,2887



DUPLICATE CUSTOMER-SNAPSHOT RECORDS: 0

FEATURE PREVIEW


,Customer_Name,Order_Frequency,Total_Sales,Total_Profit,Average_Order_Value,Average_Order_Profit,Segment,Recency_Days,Median_Order_Interval,Average_Discount,Maximum_Discount,Unique_Products,Unique_Categories,Unique_Subcategories,Repeat_Within_90_Days,Snapshot_Date
0,Aaron Hawkins,4,648.0,188.0,162.0,47.0,Corporate,91,67.0,0.085714,0.2,7,3,6,0,2016-03-31
1,Aaron Smayling,1,66.0,32.0,66.0,32.0,Corporate,248,NaN,0.000000,0.0,1,1,1,0,2016-03-31
2,Adam Hart,1,5.0,2.0,5.0,2.0,Corporate,136,NaN,0.000000,0.0,1,1,1,0,2016-03-31
3,Adam Shillingsburg,2,708.0,130.0,354.0,65.0,Consumer,98,93.0,0.060000,0.2,5,2,5,0,2016-03-31
4,Adrian Barton,1,1108.0,499.0,1108.0,499.0,Consumer,102,NaN,0.000000,0.0,2,1,2,0,2016-03-31


##  — CHRONOLOGICAL DATA SPLITTING

In [ ]:
# Defining snapshot periods
train_dates = snapshot_dates[:5]
validation_dates = snapshot_dates[5:8]
test_dates = snapshot_dates[8:]

# Creating chronological datasets
train_df = signal_model_df[
    signal_model_df["Snapshot_Date"].isin(train_dates)
].copy()

validation_df = signal_model_df[
    signal_model_df["Snapshot_Date"].isin(validation_dates)
].copy()

test_df = signal_model_df[
    signal_model_df["Snapshot_Date"].isin(test_dates)
].copy()

# Handling missing historical intervals
# Keeping the original information that an interval was unavailable.
for dataset in [train_df, validation_df, test_df]:

    dataset["Interval_Unavailable"] = (
        dataset["Median_Order_Interval"].isna().astype(int)
    )

# Feature definitions
numeric_features = [
    "Order_Frequency",
    "Total_Sales",
    "Total_Profit",
    "Average_Order_Value",
    "Average_Order_Profit",
    "Recency_Days",
    "Median_Order_Interval",
    "Average_Discount",
    "Maximum_Discount",
    "Unique_Products",
    "Unique_Categories",
    "Unique_Subcategories",
    "Interval_Unavailable"
]

categorical_features = ["Segment"]

target = "Repeat_Within_90_Days"

# Dataset audit
print("CHRONOLOGICAL SPLIT AUDIT")
print("=" * 65)

for name, dataset in [
    ("TRAINING", train_df),
    ("VALIDATION", validation_df),
    ("TEST", test_df)
]:

    print(f"\n{name}")
    print("-" * 40)

    print("Rows:", len(dataset))
    print("Unique customers:", dataset["Customer_Name"].nunique())
    print("Snapshots:",
          dataset["Snapshot_Date"].dt.strftime("%Y-%m-%d").unique())

    print("Repeat rate:",
          round(dataset[target].mean() * 100, 2), "%")

    print("Missing intervals:",
          dataset["Median_Order_Interval"].isna().sum())

    print("Missing values in other numeric features:",
          dataset[numeric_features]
          .drop(columns=["Median_Order_Interval"])
          .isna().sum().sum())

# Customer overlap audit
train_customers = set(train_df["Customer_Name"])
validation_customers = set(validation_df["Customer_Name"])
test_customers = set(test_df["Customer_Name"])

print("\nCUSTOMER OVERLAP AUDIT")
print("=" * 65)

print("Train ∩ Validation:",
      len(train_customers & validation_customers))

print("Train ∩ Test:",
      len(train_customers & test_customers))

print("Validation ∩ Test:",
      len(validation_customers & test_customers))

# Confirming that the target and identifiers are not feature columns
excluded_columns = [
    "Customer_Name",
    "Snapshot_Date",
    target
]

print("\nFEATURE COUNT:",
      len(numeric_features) + len(categorical_features))

print("EXCLUDED FROM MODEL FEATURES:", excluded_columns)

print("\nTRAINING TARGET DISTRIBUTION")
display(train_df[target].value_counts(normalize=True).mul(100).round(2))

print("\nVALIDATION TARGET DISTRIBUTION")
display(validation_df[target].value_counts(normalize=True).mul(100).round(2))

print("\nTEST TARGET DISTRIBUTION")
display(test_df[target].value_counts(normalize=True).mul(100).round(2))


CHRONOLOGICAL SPLIT AUDIT

TRAINING
----------------------------------------
Rows: 2885
Unique customers: 733
Snapshots: ['2016-03-31' '2016-06-30' '2016-09-30' '2016-12-31' '2017-03-31']
Repeat rate: 28.53 %
Missing intervals: 1425
Missing values in other numeric features: 0

VALIDATION
----------------------------------------
Rows: 1891
Unique customers: 741
Snapshots: ['2017-06-30' '2017-09-30' '2017-12-31']
Repeat rate: 35.59 %
Missing intervals: 805
Missing values in other numeric features: 0

TEST
----------------------------------------
Rows: 1967
Unique customers: 728
Snapshots: ['2018-03-31' '2018-06-30' '2018-09-30']
Repeat rate: 45.65 %
Missing intervals: 657
Missing values in other numeric features: 0

CUSTOMER OVERLAP AUDIT
Train ∩ Validation: 694
Train ∩ Test: 672
Validation ∩ Test: 692

FEATURE COUNT: 14
EXCLUDED FROM MODEL FEATURES: ['Customer_Name', 'Snapshot_Date', 'Repeat_Within_90_Days']

TRAINING TARGET DISTRIBUTION


,proportion
Repeat_Within_90_Days,
0,71.47
1,28.53



VALIDATION TARGET DISTRIBUTION


,proportion
Repeat_Within_90_Days,
0,64.41
1,35.59



TEST TARGET DISTRIBUTION


,proportion
Repeat_Within_90_Days,
0,54.35
1,45.65


## — BASELINE VS LOGISTIC REGRESSION


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

#   Preparing feature matrices
X_train = train_df[numeric_features + categorical_features]
y_train = train_df[target]

X_val = validation_df[numeric_features + categorical_features]
y_val = validation_df[target]

#    Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

#    Categorical preprocessing
categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

#     Combining preprocessing
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

#    Building logistic regression pipeline
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

#      Fiting using training data only
logistic_model.fit(X_train, y_train)

#      Baseline model
baseline_model = DummyClassifier(strategy="prior")
baseline_model.fit(X_train, y_train)

#      Validation predictions
logistic_predictions = logistic_model.predict(X_val)
logistic_probabilities = logistic_model.predict_proba(X_val)[:, 1]

baseline_predictions = baseline_model.predict(X_val)
baseline_probabilities = baseline_model.predict_proba(X_val)[:, 1]

#     Evaluation function
def evaluate_model(name, y_true, predictions, probabilities):

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1_Score": f1_score(
            y_true, predictions, zero_division=0
        ),
        "ROC_AUC": roc_auc_score(y_true, probabilities),
        "PR_AUC": average_precision_score(
            y_true, probabilities
        )
    }

validation_results = pd.DataFrame([
    evaluate_model(
        "Dummy Baseline",
        y_val,
        baseline_predictions,
        baseline_probabilities
    ),
    evaluate_model(
        "Logistic Regression",
        y_val,
        logistic_predictions,
        logistic_probabilities
    )
])

print("VALIDATION PERFORMANCE")
print("=" * 65)
display(validation_results.round(4))

print("\nLOGISTIC REGRESSION CONFUSION MATRIX")
print(confusion_matrix(y_val, logistic_predictions))

print("\nLOGISTIC REGRESSION CLASSIFICATION REPORT")
print(classification_report(
    y_val,
    logistic_predictions,
    zero_division=0
))

print("\nBASELINE CONFUSION MATRIX")
print(confusion_matrix(y_val, baseline_predictions))


VALIDATION PERFORMANCE


,Model,Accuracy,Precision,Recall,F1_Score,ROC_AUC,PR_AUC
0,Dummy Baseline,0.6441,0.0,0.0,0.0,0.5000,0.3559
1,Logistic Regression,0.6441,0.0,0.0,0.0,0.5158,0.3724



LOGISTIC REGRESSION CONFUSION MATRIX
[[1218    0]
 [ 673    0]]

LOGISTIC REGRESSION CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.64      1.00      0.78      1218
           1       0.00      0.00      0.00       673

    accuracy                           0.64      1891
   macro avg       0.32      0.50      0.39      1891
weighted avg       0.41      0.64      0.50      1891


BASELINE CONFUSION MATRIX
[[1218    0]
 [ 673    0]]


# — PROBABILITY AND THRESHOLD DIAGNOSTICS

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

#   Probability distribution
probability_df = pd.DataFrame({
    "Actual_Class": y_val.values,
    "Predicted_Probability": logistic_probabilities
})

print("PREDICTED PROBABILITY DISTRIBUTION")
print("=" * 60)

print("\nOverall:")
display(
    probability_df["Predicted_Probability"]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .to_frame("Probability")
    .round(4)
)

print("\nBy actual outcome:")
display(
    probability_df.groupby("Actual_Class")
    ["Predicted_Probability"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(4)
)

#   Testing different thresholds on validation data
threshold_results = []

for threshold in [0.05, 0.10, 0.15, 0.20, 0.25,
                  0.30, 0.35, 0.40, 0.45, 0.50]:

    predictions = (
        logistic_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Predicted_Repeat_Count": int(predictions.sum()),
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1_Score": f1_score(
            y_val, predictions, zero_division=0
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_val, predictions
        )
    })

threshold_df = pd.DataFrame(threshold_results)

print("\nTHRESHOLD COMPARISON")
print("=" * 60)
display(threshold_df.round(4))

#    Probability separation
print("\nPROBABILITY SEPARATION")
print("=" * 60)

for actual_class, label in [
    (0, "Actual non-repeat purchasers"),
    (1, "Actual repeat purchasers")
]:

    values = probability_df.loc[
        probability_df["Actual_Class"] == actual_class,
        "Predicted_Probability"
    ]

    print(
        f"{label}: "
        f"median={values.median():.4f}, "
        f"mean={values.mean():.4f}"
    )


PREDICTED PROBABILITY DISTRIBUTION

Overall:


,Probability
count,1891.0000
mean,0.2821
std,0.0264
min,0.1180
10%,0.2519
25%,0.2671
50%,0.2831
75%,0.2988
90%,0.3135
95%,0.3221



By actual outcome:


,count,mean,median,min,max
Actual_Class,,,,,
0,1218,0.2819,0.2826,0.1571,0.3675
1,673,0.2826,0.2845,0.1180,0.3565



THRESHOLD COMPARISON


,Threshold,Predicted_Repeat_Count,Precision,Recall,F1_Score,Balanced_Accuracy
0,0.05,1891,0.3559,1.0000,0.5250,0.5000
1,0.10,1891,0.3559,1.0000,0.5250,0.5000
2,0.15,1889,0.3552,0.9970,0.5238,0.4985
3,0.20,1873,0.3545,0.9866,0.5216,0.4970
4,0.25,1723,0.3523,0.9019,0.5067,0.4928
5,0.30,441,0.3832,0.2511,0.3034,0.5139
6,0.35,10,0.5000,0.0074,0.0146,0.5017
7,0.40,0,0.0000,0.0000,0.0000,0.5000
8,0.45,0,0.0000,0.0000,0.0000,0.5000
9,0.50,0,0.0000,0.0000,0.0000,0.5000



PROBABILITY SEPARATION
Actual non-repeat purchasers: median=0.2826, mean=0.2819
Actual repeat purchasers: median=0.2845, mean=0.2826


# — LOGISTIC REGRESSION DIAGNOSTICS

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

#  Predictions on training data
train_probabilities = logistic_model.predict_proba(X_train)[:, 1]
train_predictions = logistic_model.predict(X_train)

#  Comparing training and validation performance
diagnostic_results = pd.DataFrame([
    {
        "Dataset": "Training",
        "Positive_Rate": y_train.mean(),
        "Mean_Predicted_Probability": train_probabilities.mean(),
        "Probability_Std": train_probabilities.std(),
        "ROC_AUC": roc_auc_score(y_train, train_probabilities),
        "PR_AUC": average_precision_score(y_train, train_probabilities)
    },
    {
        "Dataset": "Validation",
        "Positive_Rate": y_val.mean(),
        "Mean_Predicted_Probability": logistic_probabilities.mean(),
        "Probability_Std": logistic_probabilities.std(),
        "ROC_AUC": roc_auc_score(y_val, logistic_probabilities),
        "PR_AUC": average_precision_score(y_val, logistic_probabilities)
    }
])

print("TRAINING VS VALIDATION")
print("=" * 65)
display(diagnostic_results.round(4))

#  Extracting logistic regression coefficients
feature_names = (
    logistic_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    logistic_model
    .named_steps["classifier"]
    .coef_[0]
)

coefficient_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

coefficient_df["Absolute_Coefficient"] = (
    coefficient_df["Coefficient"].abs()
)

coefficient_df = coefficient_df.sort_values(
    "Absolute_Coefficient",
    ascending=False
)

print("\nMOST INFLUENTIAL MODEL COEFFICIENTS")
print("=" * 65)
display(coefficient_df.head(15).round(4))

#  Comparing historical feature relationships with the target
feature_target_correlations = (
    train_df[numeric_features + [target]]
    .corr(numeric_only=True)[target]
    .drop(target)
    .sort_values(key=abs, ascending=False)
)

print("\nNUMERIC FEATURE-TARGET CORRELATIONS IN TRAINING")
print("=" * 65)
display(feature_target_correlations.to_frame(
    "Correlation_with_Repeat_Target"
).round(4))


TRAINING VS VALIDATION


,Dataset,Positive_Rate,Mean_Predicted_Probability,Probability_Std,ROC_AUC,PR_AUC
0,Training,0.2853,0.2853,0.0283,0.5410,0.3195
1,Validation,0.3559,0.2821,0.0264,0.5158,0.3724



MOST INFLUENTIAL MODEL COEFFICIENTS


,Feature,Coefficient,Absolute_Coefficient
4,numeric__Average_Order_Profit,0.1840,0.1840
11,numeric__Unique_Subcategories,0.1594,0.1594
9,numeric__Unique_Products,-0.1453,0.1453
2,numeric__Total_Profit,-0.0999,0.0999
5,numeric__Recency_Days,0.0711,0.0711
14,categorical__Segment_Corporate,-0.0612,0.0612
13,categorical__Segment_Consumer,0.0539,0.0539
1,numeric__Total_Sales,-0.0513,0.0513
0,numeric__Order_Frequency,0.0472,0.0472
12,numeric__Interval_Unavailable,0.0367,0.0367



NUMERIC FEATURE-TARGET CORRELATIONS IN TRAINING


,Correlation_with_Repeat_Target
Recency_Days,0.0290
Average_Order_Profit,0.0246
Interval_Unavailable,0.0161
Order_Frequency,-0.0152
Unique_Products,-0.0151
Total_Sales,-0.0138
Unique_Categories,-0.0092
Total_Profit,0.0083
Unique_Subcategories,-0.0078
Average_Discount,0.0077


In [ ]:

# — RANDOM FOREST EXPERIMENT

from sklearn.ensemble import RandomForestClassifier

# Standard Random Forest
rf_standard = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=400,
        max_depth=8,
        min_samples_split=20,
        min_samples_leaf=10,
        max_features="sqrt",
        class_weight=None,
        random_state=42,
        n_jobs=-1
    ))
])

# Class-weighted Random Forest
rf_balanced = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=400,
        max_depth=8,
        min_samples_split=20,
        min_samples_leaf=10,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

#  Fitting only on training data
rf_standard.fit(X_train, y_train)
rf_balanced.fit(X_train, y_train)

#  Evaluation helper
def evaluate_rf(name, model):

    train_prob = model.predict_proba(X_train)[:, 1]
    val_prob = model.predict_proba(X_val)[:, 1]

    train_pred = (train_prob >= 0.5).astype(int)
    val_pred = (val_prob >= 0.5).astype(int)

    return {
        "Model": name,
        "Train_ROC_AUC": roc_auc_score(y_train, train_prob),
        "Validation_ROC_AUC": roc_auc_score(y_val, val_prob),
        "Train_PR_AUC": average_precision_score(y_train, train_prob),
        "Validation_PR_AUC": average_precision_score(y_val, val_prob),
        "Validation_Precision": precision_score(
            y_val, val_pred, zero_division=0
        ),
        "Validation_Recall": recall_score(
            y_val, val_pred, zero_division=0
        ),
        "Validation_F1": f1_score(
            y_val, val_pred, zero_division=0
        )
    }

rf_results = pd.DataFrame([
    evaluate_rf("Standard Random Forest", rf_standard),
    evaluate_rf("Balanced Random Forest", rf_balanced)
])

print("RANDOM FOREST PERFORMANCE")
print("=" * 70)
display(rf_results.round(4))

#  Storing validation probabilities for later threshold analysis
rf_standard_val_prob = rf_standard.predict_proba(X_val)[:, 1]
rf_balanced_val_prob = rf_balanced.predict_proba(X_val)[:, 1]

#  Comparing probability distributions
rf_probability_summary = pd.DataFrame({
    "Actual_Class": y_val.values,
    "Standard_RF": rf_standard_val_prob,
    "Balanced_RF": rf_balanced_val_prob
})

print("\nPREDICTED PROBABILITY DISTRIBUTIONS")
print("=" * 70)

for model_name in ["Standard_RF", "Balanced_RF"]:

    print(f"\n{model_name}")

    display(
        rf_probability_summary.groupby("Actual_Class")[model_name]
        .agg(["mean", "median", "std", "min", "max"])
        .round(4)
    )

#  Featuring importance from standard RF
rf_feature_names = (
    rf_standard
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

rf_importances = (
    rf_standard
    .named_steps["classifier"]
    .feature_importances_
)

rf_importance_df = pd.DataFrame({
    "Feature": rf_feature_names,
    "Importance": rf_importances
}).sort_values("Importance", ascending=False)

print("\nSTANDARD RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 70)
display(rf_importance_df.head(15).round(4))


RANDOM FOREST PERFORMANCE


,Model,Train_ROC_AUC,Validation_ROC_AUC,Train_PR_AUC,Validation_PR_AUC,Validation_Precision,Validation_Recall,Validation_F1
0,Standard Random Forest,0.8576,0.5272,0.6879,0.3810,0.00,0.000,0.0000
1,Balanced Random Forest,0.8580,0.5266,0.7037,0.3831,0.38,0.266,0.3129



PREDICTED PROBABILITY DISTRIBUTIONS

Standard_RF


,mean,median,std,min,max
Actual_Class,,,,,
0,0.2801,0.2784,0.0394,0.1515,0.4793
1,0.2842,0.2821,0.0407,0.1700,0.4641



Balanced_RF


,mean,median,std,min,max
Actual_Class,,,,,
0,0.4639,0.4705,0.0514,0.2790,0.6392
1,0.4686,0.4753,0.0530,0.2769,0.6405



STANDARD RANDOM FOREST FEATURE IMPORTANCE


,Feature,Importance
5,numeric__Recency_Days,0.1574
4,numeric__Average_Order_Profit,0.1303
2,numeric__Total_Profit,0.1255
1,numeric__Total_Sales,0.1199
3,numeric__Average_Order_Value,0.1143
7,numeric__Average_Discount,0.0831
6,numeric__Median_Order_Interval,0.0771
9,numeric__Unique_Products,0.0442
11,numeric__Unique_Subcategories,0.0395
8,numeric__Maximum_Discount,0.0336


# — TEMPORAL DISTRIBUTION SHIFT AUDIT

In [ ]:
#  Targeting rate across each snapshot
snapshot_audit = (
    signal_model_df
    .groupby("Snapshot_Date")
    .agg(
        Customers=("Repeat_Within_90_Days", "size"),
        Repeat_Rate=("Repeat_Within_90_Days", "mean"),
        Repeat_Customers=("Repeat_Within_90_Days", "sum")
    )
    .reset_index()
)

snapshot_audit["Period"] = np.where(
    snapshot_audit["Snapshot_Date"].isin(train_dates),
    "Training",
    np.where(
        snapshot_audit["Snapshot_Date"].isin(validation_dates),
        "Validation",
        "Test"
    )
)

print("REPEAT-PURCHASE RATE BY SNAPSHOT")
print("=" * 75)
display(snapshot_audit.round(4))

#  Comparing numerical feature distributions
shift_rows = []

for feature in numeric_features:

    train_values = train_df[feature].dropna()
    val_values = validation_df[feature].dropna()

    train_mean = train_values.mean()
    val_mean = val_values.mean()

    train_std = train_values.std()
    val_std = val_values.std()

    pooled_std = np.sqrt(
        (train_std**2 + val_std**2) / 2
    )

    smd = (
        (val_mean - train_mean) / pooled_std
        if pooled_std > 0 else 0
    )

    shift_rows.append({
        "Feature": feature,
        "Train_Mean": train_mean,
        "Validation_Mean": val_mean,
        "Train_Median": train_values.median(),
        "Validation_Median": val_values.median(),
        "Standardized_Mean_Difference": smd,
        "Absolute_SMD": abs(smd)
    })

feature_shift_df = pd.DataFrame(shift_rows)

feature_shift_df = feature_shift_df.sort_values(
    "Absolute_SMD",
    ascending=False
)

print("\nNUMERICAL FEATURE DISTRIBUTION SHIFT")
print("=" * 75)
display(feature_shift_df.round(4))

#   Comparing categorical segment distributions
segment_shift = pd.crosstab(
    signal_model_df["Segment"],
    signal_model_df["Snapshot_Date"].map(
        lambda d: (
            "Training" if d in train_dates
            else "Validation" if d in validation_dates
            else "Test"
        )
    ),
    normalize="columns"
)

print("\nCUSTOMER SEGMENT DISTRIBUTION")
print("=" * 75)
display(segment_shift.round(4))

#   Missing interval proportions
missingness_audit = pd.DataFrame({
    "Dataset": ["Training", "Validation", "Test"],
    "Missing_Interval_Rate": [
        train_df["Median_Order_Interval"].isna().mean(),
        validation_df["Median_Order_Interval"].isna().mean(),
        test_df["Median_Order_Interval"].isna().mean()
    ]
})

print("\nMISSING INTERVAL RATE")
print("=" * 75)
display(missingness_audit.round(4))


REPEAT-PURCHASE RATE BY SNAPSHOT


,Snapshot_Date,Customers,Repeat_Rate,Repeat_Customers,Period
0,2016-03-31,579,0.2418,140,Training
1,2016-06-30,576,0.3056,176,Training
2,2016-09-30,568,0.3979,226,Training
3,2016-12-31,573,0.2007,115,Training
4,2017-03-31,589,0.2818,166,Training
5,2017-06-30,622,0.3698,230,Validation
6,2017-09-30,631,0.4216,266,Validation
7,2017-12-31,638,0.2774,177,Validation
8,2018-03-31,645,0.3721,240,Test
9,2018-06-30,656,0.4405,289,Test



NUMERICAL FEATURE DISTRIBUTION SHIFT


,Feature,Train_Mean,Validation_Mean,Train_Median,Validation_Median,Standardized_Mean_Difference,Absolute_SMD
0,Order_Frequency,1.7626,1.9699,2.00,2.00,0.2032,0.2032
12,Interval_Unavailable,0.4939,0.4257,0.00,0.00,-0.1372,0.1372
5,Recency_Days,139.0922,125.7054,121.00,100.00,-0.1328,0.1328
6,Median_Order_Interval,106.6377,97.5262,93.00,82.50,-0.1242,0.1242
9,Unique_Products,3.5695,3.8948,3.00,3.00,0.1140,0.1140
11,Unique_Subcategories,2.9723,3.1660,3.00,3.00,0.0990,0.0990
1,Total_Sales,829.2250,902.4907,393.00,448.00,0.0563,0.0563
10,Unique_Categories,1.8215,1.8636,2.00,2.00,0.0550,0.0550
3,Average_Order_Value,474.2515,443.4716,244.00,246.50,-0.0447,0.0447
8,Maximum_Discount,0.2810,0.2917,0.20,0.20,0.0403,0.0403



CUSTOMER SEGMENT DISTRIBUTION


Snapshot_Date,Test,Training,Validation
Segment,,,
Consumer,0.516,0.5137,0.5172
Corporate,0.304,0.3002,0.2930
Home Office,0.180,0.1861,0.1898



MISSING INTERVAL RATE


,Dataset,Missing_Interval_Rate
0,Training,0.4939
1,Validation,0.4257
2,Test,0.3340


# — TEMPORAL FEATURE ENGINEERING

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

#   Creating separate copies; preserve original datasets
train_time_df = train_df.copy()
validation_time_df = validation_df.copy()

def add_calendar_features(data):

    data = data.copy()

    month = data["Snapshot_Date"].dt.month

    # Cyclical representation of calendar month
    data["Snapshot_Month_Sin"] = np.sin(
        2 * np.pi * month / 12
    )

    data["Snapshot_Month_Cos"] = np.cos(
        2 * np.pi * month / 12
    )

    # Linear year trend
    data["Snapshot_Year_Trend"] = (
        data["Snapshot_Date"].dt.year - 2016
    )

    return data

train_time_df = add_calendar_features(train_time_df)
validation_time_df = add_calendar_features(validation_time_df)

temporal_numeric_features = numeric_features + [
    "Snapshot_Month_Sin",
    "Snapshot_Month_Cos",
    "Snapshot_Year_Trend"
]

# Building a fresh preprocessing pipeline
temporal_preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        temporal_numeric_features
    ),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features
    )
])

temporal_logistic_model = Pipeline([
    ("preprocessor", temporal_preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

X_train_time = train_time_df[
    temporal_numeric_features + categorical_features
]

X_val_time = validation_time_df[
    temporal_numeric_features + categorical_features
]

# Fitting on training data only
temporal_logistic_model.fit(X_train_time, y_train)

# Predictions
train_time_prob = temporal_logistic_model.predict_proba(
    X_train_time
)[:, 1]

val_time_prob = temporal_logistic_model.predict_proba(
    X_val_time
)[:, 1]

# Comparing against original logistic regression
comparison = pd.DataFrame([
    {
        "Model": "Original Logistic Regression",
        "Train_ROC_AUC": roc_auc_score(
            y_train, train_probabilities
        ),
        "Validation_ROC_AUC": roc_auc_score(
            y_val, logistic_probabilities
        ),
        "Validation_PR_AUC": average_precision_score(
            y_val, logistic_probabilities
        ),
        "Validation_Brier": brier_score_loss(
            y_val, logistic_probabilities
        )
    },
    {
        "Model": "Temporal Logistic Regression",
        "Train_ROC_AUC": roc_auc_score(
            y_train, train_time_prob
        ),
        "Validation_ROC_AUC": roc_auc_score(
            y_val, val_time_prob
        ),
        "Validation_PR_AUC": average_precision_score(
            y_val, val_time_prob
        ),
        "Validation_Brier": brier_score_loss(
            y_val, val_time_prob
        )
    }
])

print("TEMPORAL FEATURE EXPERIMENT")
print("=" * 75)
display(comparison.round(4))

# Checking predicted probability separation
temporal_probability_summary = pd.DataFrame({
    "Actual_Class": y_val.values,
    "Predicted_Probability": val_time_prob
})

print("\nVALIDATION PROBABILITIES BY ACTUAL CLASS")
print("=" * 75)

display(
    temporal_probability_summary
    .groupby("Actual_Class")["Predicted_Probability"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(4)
)


TEMPORAL FEATURE EXPERIMENT


,Model,Train_ROC_AUC,Validation_ROC_AUC,Validation_PR_AUC,Validation_Brier
0,Original Logistic Regression,0.5410,0.5158,0.3724,0.2351
1,Temporal Logistic Regression,0.5894,0.5682,0.4039,0.2283



VALIDATION PROBABILITIES BY ACTUAL CLASS


,count,mean,median,std,min,max
Actual_Class,,,,,,
0,1218,0.3900,0.4175,0.0722,0.1623,0.5381
1,673,0.4068,0.4301,0.0691,0.1352,0.5533


# — TEMPORAL MODEL INTERPRETATION

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

#  Inspecting temporal model coefficients
temporal_feature_names = (
    temporal_logistic_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

temporal_coefficients = (
    temporal_logistic_model
    .named_steps["classifier"]
    .coef_[0]
)

temporal_coefficient_df = pd.DataFrame({
    "Feature": temporal_feature_names,
    "Coefficient": temporal_coefficients
})

temporal_coefficient_df["Absolute_Coefficient"] = (
    temporal_coefficient_df["Coefficient"].abs()
)

temporal_coefficient_df = temporal_coefficient_df.sort_values(
    "Absolute_Coefficient",
    ascending=False
)

print("TEMPORAL MODEL — TOP COEFFICIENTS")
print("=" * 75)
display(temporal_coefficient_df.head(20).round(4))

#   Evaluating each validation snapshot separately
snapshot_results = []

for snapshot in sorted(validation_time_df["Snapshot_Date"].unique()):

    mask = (
        validation_time_df["Snapshot_Date"] == snapshot
    ).to_numpy()

    actual = y_val.iloc[np.where(mask)[0]]
    predicted_prob = val_time_prob[mask]

    snapshot_results.append({
        "Snapshot": snapshot.strftime("%Y-%m-%d"),
        "Customers": len(actual),
        "Repeat_Rate": actual.mean(),
        "ROC_AUC": (
            roc_auc_score(actual, predicted_prob)
            if actual.nunique() == 2 else np.nan
        ),
        "PR_AUC": average_precision_score(
            actual, predicted_prob
        ),
        "Brier_Score": brier_score_loss(
            actual, predicted_prob
        )
    })

snapshot_performance_df = pd.DataFrame(snapshot_results)

print("\nVALIDATION PERFORMANCE BY SNAPSHOT")
print("=" * 75)
display(snapshot_performance_df.round(4))

#  Comparing calendar-feature coefficients only
calendar_coefficients = temporal_coefficient_df[
    temporal_coefficient_df["Feature"].str.contains(
        "Snapshot_"
    )
]

print("\nCALENDAR FEATURE COEFFICIENTS")
print("=" * 75)
display(calendar_coefficients.round(4))


TEMPORAL MODEL — TOP COEFFICIENTS


,Feature,Coefficient,Absolute_Coefficient
13,numeric__Snapshot_Month_Sin,-0.2902,0.2902
4,numeric__Average_Order_Profit,0.1828,0.1828
11,numeric__Unique_Subcategories,0.1591,0.1591
15,numeric__Snapshot_Year_Trend,0.1583,0.1583
14,numeric__Snapshot_Month_Cos,-0.1560,0.1560
9,numeric__Unique_Products,-0.1456,0.1456
2,numeric__Total_Profit,-0.1004,0.1004
16,categorical__Segment_Consumer,0.0606,0.0606
1,numeric__Total_Sales,-0.0587,0.0587
17,categorical__Segment_Corporate,-0.0586,0.0586



VALIDATION PERFORMANCE BY SNAPSHOT


,Snapshot,Customers,Repeat_Rate,ROC_AUC,PR_AUC,Brier_Score
0,2017-06-30,622,0.3698,0.4965,0.3912,0.2379
1,2017-09-30,631,0.4216,0.5141,0.4251,0.2461
2,2017-12-31,638,0.2774,0.5138,0.3257,0.2013



CALENDAR FEATURE COEFFICIENTS


,Feature,Coefficient,Absolute_Coefficient
13,numeric__Snapshot_Month_Sin,-0.2902,0.2902
15,numeric__Snapshot_Year_Trend,0.1583,0.1583
14,numeric__Snapshot_Month_Cos,-0.1560,0.1560


# — CALENDAR FEATURE ABLATION

In [ ]:
from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

# Defining four controlled experiments
feature_experiments = {
    "Baseline": numeric_features,
    "Month Only": numeric_features + [
        "Snapshot_Month_Sin",
        "Snapshot_Month_Cos"
    ],
    "Year Only": numeric_features + [
        "Snapshot_Year_Trend"
    ],
    "Month + Year": temporal_numeric_features
}

ablation_models = {}
ablation_results = []
snapshot_ablation_results = []

for experiment_name, experiment_numeric_features in feature_experiments.items():

    experiment_preprocessor = ColumnTransformer([
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            experiment_numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ])

    experiment_model = Pipeline([
        ("preprocessor", experiment_preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ])

    experiment_X_train = train_time_df[
        experiment_numeric_features + categorical_features
    ]

    experiment_X_val = validation_time_df[
        experiment_numeric_features + categorical_features
    ]

    experiment_model.fit(experiment_X_train, y_train)

    train_prob = experiment_model.predict_proba(
        experiment_X_train
    )[:, 1]

    val_prob = experiment_model.predict_proba(
        experiment_X_val
    )[:, 1]

    ablation_models[experiment_name] = experiment_model

    ablation_results.append({
        "Experiment": experiment_name,
        "Train_ROC_AUC": roc_auc_score(y_train, train_prob),
        "Validation_ROC_AUC": roc_auc_score(y_val, val_prob),
        "Validation_PR_AUC": average_precision_score(y_val, val_prob),
        "Validation_Brier": brier_score_loss(y_val, val_prob)
    })

    # Evaluate each experiment within each validation snapshot
    for snapshot in sorted(validation_time_df["Snapshot_Date"].unique()):

        mask = (
            validation_time_df["Snapshot_Date"] == snapshot
        ).to_numpy()

        actual = y_val.iloc[np.where(mask)[0]]
        snapshot_prob = val_prob[mask]

        snapshot_ablation_results.append({
            "Experiment": experiment_name,
            "Snapshot": snapshot.strftime("%Y-%m-%d"),
            "ROC_AUC": (
                roc_auc_score(actual, snapshot_prob)
                if actual.nunique() == 2 else np.nan
            ),
            "PR_AUC": average_precision_score(
                actual, snapshot_prob
            ),
            "Brier_Score": brier_score_loss(
                actual, snapshot_prob
            )
        })

# Overall comparison
ablation_results_df = pd.DataFrame(ablation_results)

print("OVERALL CALENDAR ABLATION RESULTS")
print("=" * 75)
display(ablation_results_df.round(4))

# Snapshot-level comparison
snapshot_ablation_df = pd.DataFrame(snapshot_ablation_results)

print("\nSNAPSHOT-LEVEL ABLATION RESULTS")
print("=" * 75)

display(
    snapshot_ablation_df.pivot(
        index="Experiment",
        columns="Snapshot",
        values="ROC_AUC"
    ).round(4)
)

print("\nSNAPSHOT-LEVEL PR-AUC")
print("=" * 75)

display(
    snapshot_ablation_df.pivot(
        index="Experiment",
        columns="Snapshot",
        values="PR_AUC"
    ).round(4)
)


OVERALL CALENDAR ABLATION RESULTS


,Experiment,Train_ROC_AUC,Validation_ROC_AUC,Validation_PR_AUC,Validation_Brier
0,Baseline,0.5410,0.5158,0.3724,0.2351
1,Month Only,0.5804,0.5627,0.3986,0.2285
2,Year Only,0.5413,0.5162,0.3726,0.2354
3,Month + Year,0.5894,0.5682,0.4039,0.2283



SNAPSHOT-LEVEL ABLATION RESULTS


Snapshot,2017-06-30,2017-09-30,2017-12-31
Experiment,,,
Baseline,0.4874,0.5134,0.5225
Month + Year,0.4965,0.5141,0.5138
Month Only,0.4964,0.5168,0.5141
Year Only,0.4881,0.5138,0.5228



SNAPSHOT-LEVEL PR-AUC


Snapshot,2017-06-30,2017-09-30,2017-12-31
Experiment,,,
Baseline,0.3724,0.4268,0.3184
Month + Year,0.3912,0.4251,0.3257
Month Only,0.3892,0.4262,0.3234
Year Only,0.3736,0.4267,0.3186


# — CUSTOMER-LEVEL BOOTSTRAP COMPARISON

In [ ]:
# Preparing validation predictions from all three models
month_features = feature_experiments["Month Only"]
month_year_features = feature_experiments["Month + Year"]

X_val_month = validation_time_df[
    month_features + categorical_features
]

X_val_month_year = validation_time_df[
    month_year_features + categorical_features
]

prob_baseline = logistic_probabilities
prob_month = ablation_models["Month Only"].predict_proba(
    X_val_month
)[:, 1]
prob_month_year = ablation_models["Month + Year"].predict_proba(
    X_val_month_year
)[:, 1]

# Aligning all observations
bootstrap_df = pd.DataFrame({
    "Customer": validation_time_df["Customer_Name"].values,
    "Actual": y_val.values,
    "Baseline": prob_baseline,
    "Month_Only": prob_month,
    "Month_Year": prob_month_year
})

# Grouping row positions by customer
customer_groups = {
    customer: np.flatnonzero(
        bootstrap_df["Customer"].values == customer
    )
    for customer in bootstrap_df["Customer"].unique()
}

unique_customers = np.array(list(customer_groups.keys()))

# Bootstrap customers, retaining all their validation snapshots
rng = np.random.default_rng(42)
n_bootstrap = 1000

bootstrap_metrics = []

for iteration in range(n_bootstrap):

    sampled_customers = rng.choice(
        unique_customers,
        size=len(unique_customers),
        replace=True
    )

    sampled_indices = np.concatenate([
        customer_groups[customer]
        for customer in sampled_customers
    ])

    sample = bootstrap_df.iloc[sampled_indices]

    actual = sample["Actual"].to_numpy()

    if len(np.unique(actual)) < 2:
        continue

    scores = {}

    for model_name in ["Baseline", "Month_Only", "Month_Year"]:

        probabilities = sample[model_name].to_numpy()

        scores[model_name] = {
            "ROC_AUC": roc_auc_score(actual, probabilities),
            "PR_AUC": average_precision_score(
                actual, probabilities
            )
        }

    bootstrap_metrics.append({
        "Month_vs_Baseline_ROC": (
            scores["Month_Only"]["ROC_AUC"]
            - scores["Baseline"]["ROC_AUC"]
        ),
        "Month_vs_Baseline_PR": (
            scores["Month_Only"]["PR_AUC"]
            - scores["Baseline"]["PR_AUC"]
        ),
        "MonthYear_vs_Month_ROC": (
            scores["Month_Year"]["ROC_AUC"]
            - scores["Month_Only"]["ROC_AUC"]
        ),
        "MonthYear_vs_Month_PR": (
            scores["Month_Year"]["PR_AUC"]
            - scores["Month_Only"]["PR_AUC"]
        )
    })

bootstrap_results = pd.DataFrame(bootstrap_metrics)

# Summarising paired differences and percentile confidence intervals
comparison_rows = []

for metric in bootstrap_results.columns:

    values = bootstrap_results[metric]

    comparison_rows.append({
        "Comparison": metric,
        "Mean_Difference": values.mean(),
        "CI_Lower_95": values.quantile(0.025),
        "CI_Upper_95": values.quantile(0.975),
        "Bootstrap_Samples": len(values)
    })

bootstrap_summary = pd.DataFrame(comparison_rows)

print("CUSTOMER-LEVEL BOOTSTRAP RESULTS")
print("=" * 75)
display(bootstrap_summary.round(4))


CUSTOMER-LEVEL BOOTSTRAP RESULTS


,Comparison,Mean_Difference,CI_Lower_95,CI_Upper_95,Bootstrap_Samples
0,Month_vs_Baseline_ROC,0.0477,0.0237,0.0715,1000
1,Month_vs_Baseline_PR,0.0266,0.0083,0.0453,1000
2,MonthYear_vs_Month_ROC,0.0056,-0.0016,0.0124,1000
3,MonthYear_vs_Month_PR,0.0054,-0.0042,0.0144,1000


# — FINAL HELD-OUT TEST EVALUATION

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


#   Combining training and validation development data
development_df = pd.concat(
    [train_time_df, validation_time_df],
    ignore_index=True
)

y_development = pd.concat(
    [y_train, y_val],
    ignore_index=True
)

#   Preparing the untouched test period
test_time_df = add_calendar_features(test_df.copy())
y_test_final = test_time_df[target]

#   Defining a reusable model builder
def build_logistic_model(features):

    final_preprocessor = ColumnTransformer([
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ])

    return Pipeline([
        ("preprocessor", final_preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ])

#   Building fair baseline and selected models
final_models = {
    "Baseline Logistic": build_logistic_model(numeric_features),
    "Month Only Logistic": build_logistic_model(
        feature_experiments["Month Only"]
    )
}

final_results = []

for model_name, model in final_models.items():

    features = (
        numeric_features
        if model_name == "Baseline Logistic"
        else feature_experiments["Month Only"]
    )

    X_development = development_df[
        features + categorical_features
    ]

    X_test_final = test_time_df[
        features + categorical_features
    ]

    # Fitting on training + validation only
    model.fit(X_development, y_development)

    test_prob = model.predict_proba(X_test_final)[:, 1]
    test_pred = (test_prob >= 0.5).astype(int)

    final_results.append({
        "Model": model_name,
        "Test_Customers": len(y_test_final),
        "Test_Repeat_Rate": y_test_final.mean(),
        "Mean_Predicted_Probability": test_prob.mean(),
        "ROC_AUC": roc_auc_score(y_test_final, test_prob),
        "PR_AUC": average_precision_score(
            y_test_final, test_prob
        ),
        "Brier_Score": brier_score_loss(
            y_test_final, test_prob
        ),
        "Accuracy_at_0.5": accuracy_score(
            y_test_final, test_pred
        ),
        "Precision_at_0.5": precision_score(
            y_test_final, test_pred, zero_division=0
        ),
        "Recall_at_0.5": recall_score(
            y_test_final, test_pred, zero_division=0
        ),
        "F1_at_0.5": f1_score(
            y_test_final, test_pred, zero_division=0
        )
    })

final_test_results = pd.DataFrame(final_results)

print("PROJECT SIGNAL — FINAL TEST RESULTS")
print("=" * 85)
display(final_test_results.round(4))


PROJECT SIGNAL — FINAL TEST RESULTS


,Model,Test_Customers,Test_Repeat_Rate,Mean_Predicted_Probability,ROC_AUC,PR_AUC,Brier_Score,Accuracy_at_0.5,Precision_at_0.5,Recall_at_0.5,F1_at_0.5
0,Baseline Logistic,1967,0.4565,0.3145,0.5034,0.4633,0.2690,0.5425,0.3333,0.0022,0.0044
1,Month Only Logistic,1967,0.4565,0.3309,0.5725,0.5104,0.2603,0.5430,0.4286,0.0033,0.0066


#  — OPERATIONAL TARGETING ANALYSIS

In [ ]:
targeting_df = validation_time_df[
    ["Customer_Name", "Snapshot_Date", target]
].copy()

targeting_df["Predicted_Probability"] = prob_month

# Ranking customers separately within each quarterly snapshot
targeting_df["Score_Rank"] = (
    targeting_df.groupby("Snapshot_Date")[
        "Predicted_Probability"
    ].rank(method="first", ascending=False, pct=True)
)

targeting_results = []

for capacity in [0.10, 0.20, 0.30]:

    selected = targeting_df[
        targeting_df["Score_Rank"] <= capacity
    ]

    actual_repeaters = targeting_df[targeting_df[target] == 1]
    captured_repeaters = selected[target].sum()

    precision = selected[target].mean()
    recall = captured_repeaters / len(actual_repeaters)
    lift = precision / targeting_df[target].mean()

    targeting_results.append({
        "Targeting_Capacity": f"Top {int(capacity * 100)}%",
        "Customers_Selected": len(selected),
        "Actual_Repeat_Purchasers_Selected": int(captured_repeaters),
        "Precision": precision,
        "Recall": recall,
        "Lift_vs_Prevalence": lift
    })

targeting_results_df = pd.DataFrame(targeting_results)

print("VALIDATION — OPERATIONAL TARGETING")
print("=" * 75)
display(targeting_results_df.round(4))


VALIDATION — OPERATIONAL TARGETING


,Targeting_Capacity,Customers_Selected,Actual_Repeat_Purchasers_Selected,Precision,Recall,Lift_vs_Prevalence
0,Top 10%,188,77,0.4096,0.1144,1.1508
1,Top 20%,377,149,0.3952,0.2214,1.1105
2,Top 30%,566,213,0.3763,0.3165,1.0574


##  — RETENTION STRATEGY

In [ ]:
retention_strategy = pd.DataFrame([
    {
        "Customer_Segment": "Champions",
        "Observed_Problem": "High historical value; maintain engagement",
        "Recommended_Action": "Loyalty recognition, early access and relevant product recommendations",
        "Discount_Policy": "Avoid unnecessary blanket discounts",
        "Success_Metric": "Repeat purchase rate and customer profit"
    },
    {
        "Customer_Segment": "At Risk",
        "Observed_Problem": "Previously frequent customers with elevated inactivity",
        "Recommended_Action": "Prioritised personalised re-engagement",
        "Discount_Policy": "Test targeted incentives only where margins permit",
        "Success_Metric": "Reactivation rate and incremental profit"
    },
    {
        "Customer_Segment": "Potential Loyalists",
        "Observed_Problem": "Recent customers with developing purchase frequency",
        "Recommended_Action": "Product discovery and relevant follow-up communication",
        "Discount_Policy": "Use product bundles before broad discounts",
        "Success_Metric": "Second and third purchase conversion"
    },
    {
        "Customer_Segment": "Needs Attention",
        "Observed_Problem": "Moderate frequency but declining recency",
        "Recommended_Action": "Personalised reminders and customer feedback",
        "Discount_Policy": "Avoid incentives until profitability is checked",
        "Success_Metric": "Return purchase rate"
    },
    {
        "Customer_Segment": "Hibernating",
        "Observed_Problem": "Low recent activity and low historical contribution",
        "Recommended_Action": "Low-cost re-engagement test",
        "Discount_Policy": "Set a strict incentive ceiling",
        "Success_Metric": "Reactivation cost versus incremental profit"
    },
    {
        "Customer_Segment": "New Customers",
        "Observed_Problem": "Limited purchase history",
        "Recommended_Action": "Onboarding and complementary product recommendations",
        "Discount_Policy": "Do not assume a discount is necessary",
        "Success_Metric": "Second purchase rate"
    },
    {
        "Customer_Segment": "Regular Customers",
        "Observed_Problem": "Moderate historical engagement",
        "Recommended_Action": "Relevant recommendations and consistent service",
        "Discount_Policy": "Margin-aware promotions",
        "Success_Metric": "Purchase frequency and profit"
    }
])

display(retention_strategy)


,Customer_Segment,Observed_Problem,Recommended_Action,Discount_Policy,Success_Metric
0,Champions,High historical value; maintain engagement,"Loyalty recognition, early access and relevant...",Avoid unnecessary blanket discounts,Repeat purchase rate and customer profit
1,At Risk,Previously frequent customers with elevated in...,Prioritised personalised re-engagement,Test targeted incentives only where margins pe...,Reactivation rate and incremental profit
2,Potential Loyalists,Recent customers with developing purchase freq...,Product discovery and relevant follow-up commu...,Use product bundles before broad discounts,Second and third purchase conversion
3,Needs Attention,Moderate frequency but declining recency,Personalised reminders and customer feedback,Avoid incentives until profitability is checked,Return purchase rate
4,Hibernating,Low recent activity and low historical contrib...,Low-cost re-engagement test,Set a strict incentive ceiling,Reactivation cost versus incremental profit
5,New Customers,Limited purchase history,Onboarding and complementary product recommend...,Do not assume a discount is necessary,Second purchase rate
6,Regular Customers,Moderate historical engagement,Relevant recommendations and consistent service,Margin-aware promotions,Purchase frequency and profit


# Final Research Conclusion

Project SIGNAL investigated customer purchasing behaviour, repeat purchasing, discount economics, profitability and retention opportunities using the Sample Superstore dataset.

The analysis covered 9,993 cleaned transaction records, 5,009 orders and 793 customer profiles across 2015–2018.

## Key Findings

1. **Customer purchasing behaviour:** Purchase frequency and total spending were positively associated with historical customer profit. However, higher purchasing activity did not automatically translate into better profit margins.

2. **Discount economics:** Higher discount levels were strongly associated with lower profit margins. The product-level fixed-effects analysis showed that this relationship remained substantial after accounting for product-specific differences. The results are observational and do not establish causation.

3. **Customer segmentation:** RFM analysis identified eight customer segments. Champions contributed 23.64% of total historical profit, while At Risk customers contributed 22.87%, despite representing only 90 customers.

4. **Retention opportunities:** Thirty-three profitable At Risk customers were classified as high-priority re-engagement candidates using their historical purchasing cadence. Their historical profit was $33,384, but this amount must not be interpreted as recoverable campaign profit.

5. **Predictive modelling:** Logistic regression with calendar-month features achieved a test ROC-AUC of 0.5725 and PR-AUC of 0.5104. This outperformed the original logistic regression baseline, although predictive discrimination remained modest.

6. **Operational targeting:** Selecting the top 10% of customers by predicted probability produced a validation lift of 1.15×. This suggests limited prioritisation value rather than strong predictive capability.

## Business Recommendations

- Prioritise profitable At Risk customers for carefully designed re-engagement experiments.
- Avoid blanket discounting, particularly where historical margins are already weak.
- Use customer-specific purchase history to guide recommendations.
- Monitor repeat purchases, reactivation rates and customer profitability together.
- Test retention campaigns with control groups before claiming incremental business impact.

## Limitations

- The dataset is a synthetic retail dataset, not verified real-world company customer data.
- Customer names were used as customer identifiers because a dedicated customer ID was unavailable.
- Observational associations cannot establish causal effects.
- Campaign costs, customer acquisition costs and marketing exposure data were unavailable.
- Temporal test observations included customers also present in training and validation data.
- The predictive model demonstrated limited discrimination and should not be treated as deployment-ready.
- The analysis covers a limited historical period and may not generalise to other retail businesses.

## Overall Conclusion

Project SIGNAL demonstrates how transaction-level data can be transformed into customer-level insights, profitability analysis, segmentation, statistical evidence and an exploratory predictive framework.

Its central finding is that customer retention decisions should consider not only purchase frequency and spending, but also profitability, purchasing cadence and the economics of incentives.

The next research stage would require real customer-level identifiers, campaign exposure records and controlled experiments to establish whether targeted retention strategies generate incremental profit.

##  — EXPORT MODELS AND DASHBOARD ARTEFACTS

In [ ]:
from pathlib import Path
import json
import joblib
import zipfile
import sys

#   Creating output directory
ARTIFACT_DIR = Path("/content/project_signal_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

#   Saving the selected final model
final_model = final_models["Month Only Logistic"]

model_path = ARTIFACT_DIR / "signal_repeat_purchase_model.joblib"
joblib.dump(final_model, model_path)

#   Saving model configuration and feature information
month_features = feature_experiments["Month Only"]

metadata = {
    "project": "Project SIGNAL",
    "model_name": "Month Only Logistic Regression",
    "prediction_target": "Repeat_Within_90_Days",
    "prediction_horizon_days": 90,
    "historical_lookback_days": 365,
    "numeric_features": month_features,
    "categorical_features": categorical_features,
    "classification_threshold": 0.5,
    "test_roc_auc": 0.5725,
    "test_pr_auc": 0.5104,
    "test_brier_score": 0.2603,
    "sklearn_version": sklearn.__version__,
    "python_version": sys.version
}

with open(ARTIFACT_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

#    Exporting analytical datasets
dashboard_data = {
    "transactions.csv": df_clean,
    "orders.csv": df_orders,
    "customers_rfm.csv": df_rfm,
    "at_risk_customers.csv": at_risk_cadence,
    "retention_strategy.csv": retention_strategy,
    "targeting_results.csv": targeting_results_df,
    "final_model_evaluation.csv": final_test_results
}

saved_files = []

for filename, dataframe in dashboard_data.items():
    if isinstance(dataframe, pd.DataFrame):
        dataframe.to_csv(
            ARTIFACT_DIR / filename,
            index=False
        )
        saved_files.append(filename)
        print(f"Saved: {filename} | Shape: {dataframe.shape}")
    else:
        print(f"Skipped: {filename} — not a DataFrame")

#    Saving requirements
requirements = f"""pandas
numpy
scikit-learn=={sklearn.__version__}
joblib
streamlit
plotly
"""

(ARTIFACT_DIR / "requirements.txt").write_text(requirements)

#    Creating artifact documentation
readme = """
PROJECT SIGNAL — DASHBOARD ARTIFACTS

signal_repeat_purchase_model.joblib:
    Final trained logistic regression pipeline.
    Includes preprocessing and classifier.

model_metadata.json:
    Model features, target, prediction horizon and evaluation.

transactions.csv:
    Cleaned transaction-level data.

orders.csv:
    Order-level analytical data.

customers_rfm.csv:
    Customer RFM scores and segments.

at_risk_customers.csv:
    At-risk customer cadence and prioritisation information.

retention_strategy.csv:
    Proposed segment-level retention recommendations.

targeting_results.csv:
    Validation operational targeting results.

final_model_evaluation.csv:
    Final held-out model evaluation.

IMPORTANT:
The predictive model is exploratory and is not deployment-ready.
Its test ROC-AUC was 0.5725.
Do not interpret predictions as guaranteed customer behaviour.
"""

(ARTIFACT_DIR / "README.txt").write_text(readme)

#   Creating ZIP archive
zip_path = Path("/content/PROJECT_SIGNAL_ARTIFACTS.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file_path in ARTIFACT_DIR.iterdir():
        if file_path.is_file():
            archive.write(file_path, arcname=file_path.name)

#   Verifying saved files
print("\n" + "=" * 65)
print("PROJECT SIGNAL — ARTIFACT EXPORT COMPLETE")
print("=" * 65)

for file_path in sorted(ARTIFACT_DIR.iterdir()):
    if file_path.is_file():
        print(f"{file_path.name}: {file_path.stat().st_size / 1024:.1f} KB")

print("\nZIP archive:", zip_path)
print("ZIP size:", round(zip_path.stat().st_size / (1024 * 1024), 2), "MB")

#   Downloading archive in Google Colab
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Download the ZIP from:", zip_path)


Saved: transactions.csv | Shape: (9993, 22)
Saved: orders.csv | Shape: (5009, 12)
Saved: customers_rfm.csv | Shape: (793, 13)
Saved: at_risk_customers.csv | Shape: (90, 22)
Saved: retention_strategy.csv | Shape: (7, 5)
Saved: targeting_results.csv | Shape: (3, 6)
Saved: final_model_evaluation.csv | Shape: (2, 11)

PROJECT SIGNAL — ARTIFACT EXPORT COMPLETE
README.txt: 0.9 KB
at_risk_customers.csv: 18.2 KB
customers_rfm.csv: 62.9 KB
final_model_evaluation.csv: 0.5 KB
model_metadata.json: 0.9 KB
orders.csv: 462.6 KB
requirements.txt: 0.1 KB
retention_strategy.csv: 1.3 KB
signal_repeat_purchase_model.joblib: 5.0 KB
targeting_results.csv: 0.3 KB
transactions.csv: 2083.4 KB

ZIP archive: /content/PROJECT_SIGNAL_ARTIFACTS.zip
ZIP size: 0.6 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#  — MODEL SERIALIZATION TEST

In [ ]:
ARTIFACT_DIR = Path("/content/project_signal_artifacts")

#  Loading the saved model from disk
loaded_model = joblib.load(
    ARTIFACT_DIR / "signal_repeat_purchase_model.joblib"
)

#  Loading model metadata
with open(ARTIFACT_DIR / "model_metadata.json", "r") as f:
    loaded_metadata = json.load(f)

#  Prepare the same held-out test features
features = (
    loaded_metadata["numeric_features"]
    + loaded_metadata["categorical_features"]
)

X_test_check = test_time_df[features]

#  Generating predictions using the reloaded model
saved_predictions = loaded_model.predict_proba(
    X_test_check
)[:, 1]

# Compare with the original model predictions
original_predictions = final_models[
    "Month Only Logistic"
].predict_proba(X_test_check)[:, 1]

print("MODEL SERIALIZATION VERIFICATION")
print("=" * 55)
print("Model loaded:", loaded_model is not None)
print("Expected features:", len(features))
print("Actual feature columns:", X_test_check.shape[1])
print("Predictions generated:", len(saved_predictions))
print("All predictions finite:", np.isfinite(saved_predictions).all())
print(
    "Predictions identical:",
    np.allclose(saved_predictions, original_predictions)
)
print(
    "Maximum prediction difference:",
    np.max(np.abs(saved_predictions - original_predictions))
)
print("Metadata target:", loaded_metadata["prediction_target"])
print("Model test ROC-AUC:", loaded_metadata["test_roc_auc"])


MODEL SERIALIZATION VERIFICATION
Model loaded: True
Expected features: 16
Actual feature columns: 16
Predictions generated: 1967
All predictions finite: True
Predictions identical: True
Maximum prediction difference: 0.0
Metadata target: Repeat_Within_90_Days
Model test ROC-AUC: 0.5725
